# ⚡ 1교시: Stream Lab - Phase 1 & Phase 2

## 🎯 학습 목표
- Python만으로 실시간 데이터 처리가 가능함을 확인
- Late Event(늦게 도착하는 이벤트)가 발생하는 실제 원인 이해
- Processing Time 방식의 한계 체감
- Event Time의 필요성 인식

---
## 📋 사전 준비

### 1. 필수 도구 설치
```bash
# Docker & Docker Compose 설치 확인
docker --version
docker compose version

# Just 설치 (선택사항, 권장)
# Ubuntu/WSL
curl --proto '=https' --tlsv1.2 -sSf https://just.systems/install.sh | bash -s -- --to /usr/local/bin

# macOS
brew install just
```

### 2. 실습 저장소 클론
```bash
cd ~/workspace  # 원하는 작업 디렉토리로 이동
git clone https://github.com/your-org/stream-lab.git
cd stream-lab
```

---
## 🏗️ 아키텍처 개요

![Stream Lab Architecture](https://cdn.discordapp.com/attachments/1457516082071081045/1470232242319261860/image.png?ex=698bdd7a&is=698a8bfa&hm=2e030bae2238498cb4b162be9579f41df46f650d1ff6500fcb33a34466197182&)

---
## 📖 실습 시나리오

### 배경 스토리
여러분은 결제 시스템의 실시간 집계를 담당하는 데이터 엔지니어입니다.
**사용자별 10초 단위 결제 금액 합계**를 실시간으로 계산하여 DB에 저장해야 합니다.

### 데이터 흐름
1. **Generator**: 결제 이벤트 생성 → Kafka 전송
2. **Consumer**: Kafka 구독 → 10초 윈도우로 집계 → PostgreSQL 저장
3. **모니터링**: PostgreSQL 쿼리로 결과 비교

---
## 🔧 Justfile이란?

### 왜 Justfile을 사용하나요?

실습 중 서비스를 중단/재시작하거나 Phase를 전환할 때 다음과 같은 문제가 자주 발생합니다:

- **Flink Job 중복**: 재시작할 때마다 기존 Job을 취소하지 않으면 여러 Job이 동시에 돌아감
- **DB 잔여 데이터**: 이전 Phase의 데이터가 남아 결과가 섞임
- **Kafka 토픽 미생성**: Flink Job을 먼저 제출하면 토픽이 없어서 재시작 루프에 빠짐
- **컨테이너 재생성**: 환경 변수를 바꿔서 `docker compose up`하면 의존 서비스까지 재생성됨

Justfile은 이런 문제를 모두 처리하여, 각 Phase를 깔끔한 상태에서 시작할 수 있게 해줍니다.

### 주요 커맨드

```bash
just              # 사용 가능한 레시피 목록 보기

# === Phase별 실행 ===
just phase1       # Phase 1: 전체 초기화 → Chaos OFF → baseline_results
just phase2       # Phase 2: Chaos ON → naive_results (Phase 1 이후 실행)
just phase4       # Phase 4: 전체 초기화 → Flink + Chaos ON 비교

# === 결과 조회 ===
just query-phase1 # Phase 1 집계 결과
just query-phase2 # Phase 2 Processing Time vs Event Time 비교
just late-events  # Late Event 발생/감지 로그

# === Phase 전환 ===
just phase1-done  # Phase 1 → 2 전환 (Generator/Consumer만 중지)
just phase2-done  # Phase 2 → 4 전환 (Generator/Consumer만 중지)

# === 유틸리티 ===
just status       # 컨테이너 + Flink Job 상태 확인
just logs generator  # 특정 서비스 로그 확인
just clean        # 모든 컨테이너 + 볼륨 완전 삭제
```

---
# 🔵 Phase 1: "Python으로도 실시간 처리가 된다"

## 🎯 Phase 1 목표
정상적인 환경(데이터 지연 없음)에서 Python만으로 실시간 집계가 가능함을 확인합니다.

## ✅ 실행 단계

In [ ]:
# === Step 1: 실습 디렉토리로 이동 ===
cd ~/workspace/stream-lab

# === Step 2: Kafka와 PostgreSQL 시작 ===
docker compose up -d kafka postgres

# === Step 3: 서비스 준비 대기 ===
docker compose logs -f kafka
# # "Kafka Server started" 메시지 확인 후 Ctrl+C

### 💡 Kafka KRaft 모드란?

실습의 `compose.yml`을 보면 Kafka 환경 변수가 복잡해 보일 수 있습니다.
하지만 대부분은 [Apache Kafka Docker Hub](https://hub.docker.com/r/apache/kafka) 공식 문서에서 제공하는
**KRaft 모드 기본 설정**입니다.

**KRaft(Kafka Raft)**는 Kafka 3.3부터 도입된 새로운 메타데이터 관리 방식으로,
기존에 필요했던 **ZooKeeper 없이 Kafka만으로 클러스터를 운영**할 수 있게 해줍니다.

| 항목 | 기존 방식 (ZooKeeper) | 새 방식 (KRaft) |
|------|---------------------|----------------|
| 메타데이터 관리 | ZooKeeper | Kafka 자체 (Raft 합의 알고리즘) |
| 필요 컴포넌트 | Kafka + ZooKeeper | Kafka만 |
| 설정 복잡도 | 높음 | 낮음 |
| 운영 오버헤드 | 높음 | 낮음 |

In [ ]:
# === Step 4: Generator와 Python Consumer 시작 (Chaos OFF) ===
docker compose up -d generator python-consumer

# === Step 5: 로그 확인 ===
docker compose logs -f generator python-consumer
# # 이벤트가 생성되고 처리되는 로그를 실시간으로 확인
# # Ctrl+C로 로그 모니터링 종료 (서비스는 계속 실행됨)

### 📊 로그 예시

**Generator 로그:**
```
[NORMAL] 전송됨: {"user_id": "U3", "amount": 2341, "event_time": "2026-02-09T01:23:45.678"}
[NORMAL] 전송됨: {"user_id": "U1", "amount": 1823, "event_time": "2026-02-09T01:23:46.234"}
```

**Consumer 로그:**
```
[PROC] window_start=2026-02-09 01:23:40 user=U3 amount=2341 sum=2341 event_time=2026-02-09T01:23:45.678 proc_time=2026-02-09T01:23:45.680
[PROC] window_start=2026-02-09 01:23:40 user=U1 amount=1823 sum=1823 event_time=2026-02-09T01:23:46.234 proc_time=2026-02-09T01:23:46.236
```

관찰 포인트:
- `window_start`: 10초 단위로 내림된 윈도우 시작 시각 (01:23:40, 01:23:50, 01:24:00, ...)
- `event_time`과 `proc_time`이 거의 동일 (밀리초 단위 차이만 존재)
- `sum`: 해당 윈도우 내 사용자별 누적 합계

In [ ]:
# === Step 6: 1분 대기 후 결과 확인 ===
# # 최소 1분은 기다려야 여러 윈도우의 집계 결과를 확인할 수 있습니다.
sleep 60

# === Step 7: PostgreSQL 접속 ===
docker compose exec postgres psql -U postgres -d streamdb

### 🔍 결과 확인 쿼리

In [ ]:
# === 쿼리 1: 집계 결과 조회 (최근 20개 레코드) ===
SELECT
    window_start,
    user_id,
    total_amount,
    updated_at
FROM baseline_results
ORDER BY window_start, user_id
LIMIT 20;

### 📋 예상 결과 (Phase 1)

```
    window_start     | user_id | total_amount |         updated_at
---------------------+---------+--------------+----------------------------
 2026-02-09 01:23:40 | U1      |         3842 | 2026-02-09 01:23:50.123456
 2026-02-09 01:23:40 | U2      |         4521 | 2026-02-09 01:23:50.123456
 2026-02-09 01:23:40 | U3      |         2156 | 2026-02-09 01:23:50.123456
 2026-02-09 01:23:40 | U4      |         3218 | 2026-02-09 01:23:50.123456
 2026-02-09 01:23:40 | U5      |         1967 | 2026-02-09 01:23:50.123456
 2026-02-09 01:23:50 | U1      |         2889 | 2026-02-09 01:24:00.234567
 2026-02-09 01:23:50 | U2      |         3741 | 2026-02-09 01:24:00.234567
```

관찰 포인트:
- `window_start`: 10초 단위로 정렬됨 (01:23:40, 01:23:50, 01:24:00, ...)
- `updated_at`: 윈도우 종료 직후 시각 (window_start + 10초)
- 각 윈도우마다 U1~U5 사용자별 집계 결과가 INSERT됨

In [ ]:
# === 쿼리 2: 윈도우별 총 금액 확인 ===
SELECT
    window_start,
    COUNT(*) as user_count,
    SUM(total_amount) as total
FROM baseline_results
GROUP BY window_start
ORDER BY window_start;

### 📋 예상 결과 (윈도우별 집계)

```
    window_start     | user_count | total
---------------------+------------+-------
 2026-02-09 01:23:40 |          5 | 15704
 2026-02-09 01:23:50 |          5 | 14327
 2026-02-09 01:24:00 |          5 | 16892
```

- 각 윈도우마다 5명의 사용자(U1~U5)가 집계됨
- 윈도우별 총 금액은 10초 동안 발생한 모든 결제의 합계

In [ ]:
# === PostgreSQL 종료 ===
# # \q

---
## 🔬 Phase 1 코드 워크스루

Phase 1의 결과가 어떻게 만들어지는지 핵심 코드를 살펴보겠습니다.

### 1️⃣ 이벤트 생성 (producer.py)

Generator는 매초 결제 이벤트를 생성하여 Kafka로 전송합니다.

In [ ]:
# === src/producer.py 핵심 부분 ===

# 이벤트 생성 (매초 1개씩)
event = {
    "user_id": f"U{random.randint(1, 5)}",      # U1~U5 중 랜덤 선택
    "amount": random.randint(500, 5000),        # 500~5000원 랜덤 금액
    "event_time": datetime.now().isoformat(timespec="milliseconds"),  # 현재 시각
}

# 💡 event_time의 의미:
#   - 이벤트가 "발생한" 시각 (Event Time)
#   - ISO-8601 형식: "2026-02-09T01:23:45.678"
#   - 밀리초(milliseconds) 정밀도 (Flink TIMESTAMP(3) 타입과 호환)

# Phase 1에서는 CHAOS_ENABLED=false이므로 즉시 전송
send_event("[NORMAL]", event)

# 전송 함수
def send_event(label, event):
    payload = json.dumps(event).encode("utf-8")
    producer.produce(
        topic=KAFKA_TOPIC,
        key=event["user_id"].encode("utf-8"),   # 파티셔닝 키: 같은 유저는 같은 파티션
        value=payload,
        callback=delivery_report,
    )
    producer.poll(0)
    print(f"{label} 전송됨: {json.dumps(event, ensure_ascii=False)}")

**핵심 포인트:**
- `event_time`: 이벤트 생성 시점의 타임스탬프
- Phase 1에서는 생성 즉시 Kafka로 전송 (지연 없음)
- `key=user_id`: 같은 사용자의 이벤트는 같은 파티션으로 전송 (순서 보장)

### 2️⃣ Processing Time 윈도우 집계 (naive_consumer.py)

Consumer는 `datetime.now()` 기준으로 10초 윈도우를 관리합니다.

In [ ]:
# === src/naive_consumer.py 핵심 부분 ===

# 윈도우 계산 함수
def floor_window_start(now: datetime) -> datetime:
    """현재 시각을 10초 단위로 내림
#
#     예시:
#     01:23:47 → 01:23:40
#     01:23:52 → 01:23:50
#     01:24:03 → 01:24:00
    """
    return now.replace(
        second=(now.second // WINDOW_SECONDS) * WINDOW_SECONDS,
        microsecond=0,
    )

# 메인 루프 초기화
WINDOW_SECONDS = 10
current_window_start = floor_window_start(datetime.now())
window_sums = {}   # {user_id: total_amount} - 메모리에만 존재!

# 💡 window_sums의 특징:
#   - 메모리(dict)에만 존재하는 "휘발성" 상태
#   - 10초마다 DB로 flush 후 초기화됨
#   - 프로세스 종료 시 아직 flush 안 된 데이터는 유실됨 (의도된 결함)

### 3️⃣ 윈도우 닫기 (flush_window)

Processing Time으로 10초가 지나면 윈도우를 닫고 DB에 INSERT합니다.

In [ ]:
# === 윈도우 닫기 로직 ===

while True:
    now = datetime.now()

    # 10초가 지났는지 체크 (이벤트 도착 여부와 무관)
    while now >= current_window_start + timedelta(seconds=WINDOW_SECONDS):
        flush_window(conn, current_window_start, window_sums, RESULT_TABLE)
        window_sums = {}  # 메모리 상태 초기화
        current_window_start = current_window_start + timedelta(seconds=WINDOW_SECONDS)

    # Kafka에서 메시지 수신 (1초 timeout)
    msg = consumer.poll(POLL_TIMEOUT_SECONDS)
    # ...

# flush_window 함수
def flush_window(conn, window_start: datetime, window_sums: dict, table_name: str):
    if not window_sums:
        print(f"[FLUSH] table={table_name} window_start={window_start} rows=0 (비어있는 윈도우)")
        return

    rows = [(window_start, user_id, total_amount)
            for user_id, total_amount in window_sums.items()]
    query = (
        f"INSERT INTO {table_name} (window_start, user_id, total_amount) "
        "VALUES (%s, %s, %s)"
    )
    with conn.cursor() as cur:
        cur.executemany(query, rows)
    conn.commit()
    print(f"[FLUSH] table={table_name} window_start={window_start} rows={len(rows)}")

**핵심 포인트:**
- **Processing Time 기준**: `datetime.now()`로 윈도우 경계 판단
- **메모리 상태**: `window_sums` dict는 메모리에만 존재
- **INSERT Only**: 윈도우 닫힐 때 DB에 INSERT (UPDATE 없음)
- **상태 유실**: 프로세스 종료 시 아직 flush 안 된 집계는 사라짐

### 4️⃣ 이벤트 처리 (event_time 무시)

Consumer는 메시지를 받으면 **현재 윈도우**에 무조건 누적합니다.

In [ ]:
# === 이벤트 처리 로직 (의도된 결함!) ===

event = json.loads(msg.value().decode("utf-8"))
user_id = event.get("user_id")
amount = int(event.get("amount", 0))
event_time = event.get("event_time")

# 의도된 결함: event_time을 로그에 출력만 하고, 윈도우 분류에는 사용하지 않음!
window_sums[user_id] = window_sums.get(user_id, 0) + amount
print(
    f"[PROC] window_start={current_window_start} user={user_id} amount={amount} "
    f"sum={window_sums[user_id]} event_time={event_time} proc_time={datetime.now().isoformat()}"
)

# 💡 왜 이렇게 구현했을까?
#   - Phase 1에서는 event_time ≈ proc_time이므로 문제가 드러나지 않음
#   - Phase 2에서 Late Event 발생 시 문제가 명확히 드러남
#   - "Processing Time의 한계"를 체감하기 위한 교육적 의도

---
## 🔍 Phase 1 관찰 포인트

| 항목 | 관찰 내용 |
|------|----------|
| **윈도우 경계** | 10초마다 정확히 닫힘 (01:23:40, 01:23:50, 01:24:00, ...) |
| **집계 정확도** | 각 윈도우 내 사용자별 결제 금액 합계가 정확함 |
| **updated_at** | 윈도우 종료 직후 시간 (Processing Time) |
| **event_time vs proc_time** | 거의 동일 (밀리초 단위 차이만 존재) |
| **Late Event** | 없음 (Chaos OFF 상태) |

---
## ✅ Phase 1 정리

In [ ]:
# === Just 사용 시 ===
just phase1-done

# === 직접 명령어 실행 시 ===
docker compose stop generator python-consumer

**Phase 1에서 배운 것:**
- ✅ Python + Kafka + PostgreSQL만으로도 실시간 집계가 가능하다
- ✅ 10초 윈도우 단위로 사용자별 결제 금액을 집계할 수 있다
- ✅ Processing Time 방식으로도 정상 환경에서는 정확한 결과를 얻을 수 있다

**하지만...**
- ❓ 실제 운영 환경에서는 네트워크 지연, 시스템 부하 등으로 데이터가 늦게 도착할 수 있다
- ❓ 그렇다면 이 Python 구현은 여전히 정확할까?

→ **Phase 2**에서 확인해봅시다!

---
---
# 🔴 Phase 2: "그런데 데이터가 늦게 오면?"

## 🎯 Phase 2 목표
운영 환경에서 발생하는 **Late Event**(늦게 도착하는 이벤트) 상황을 재현하고,
Python 구현의 한계를 체감합니다.

---
## 🤔 Late Event란? 왜 발생하나요?

### 정의
**Late Event**: 이벤트가 발생한 시각(**Event Time**)과 서버에 도착한 시각(**Processing Time**) 사이에
차이가 발생하여, **원래 속한 윈도우가 이미 닫힌 후에 도착**하는 이벤트

### 발생 원인

| 원인 | 실제 사례 |
|------|----------|
| **네트워크 불안정** | 지하철 터널 진입으로 인터넷이 끊겼다가 5초 후 복구 |
| **기기 내 버퍼링** | 모바일 앱이 배터리 효율을 위해 이벤트를 모아서 전송 (Batching) |
| **시스템 부하** | 서버에 요청이 폭증하여 처리가 지연 |
| **분산 환경** | 여러 서버에서 전송 시 네트워크 경로 차이로 순서가 뒤바뀜 |

### 타임라인 예시

```
Event Time (발생 시각)     Processing Time (도착 시각)

01:23:07 결제 발생 ────┐
                      │ (네트워크 지연 5초)
                      └────> 01:23:12 서버 도착

01:23:10 윈도우 닫힘 ──┘ (이미 늦음!)

Result:
  - 윈도우 01:23:00~01:23:10 → amount 누락 (과소 집계)
  - 윈도우 01:23:10~01:23:20 → amount 추가 (과대 집계)
```

---
## 🌪️ Chaos Engineering이란?

### 정의
**Chaos Engineering**: 시스템이 예상치 못한 상황에서도 견딜 수 있는지 검증하기 위해
**의도적으로 장애를 주입**하는 엔지니어링 기법

### 유명 사례
- **Netflix Chaos Monkey**: 랜덤하게 프로덕션 서버를 종료하여 장애 복구 능력 검증
- **Amazon GameDay**: 의도적으로 AWS 서비스를 장애 상태로 만들어 대응 능력 훈련

### Stream Lab에서의 활용
- **목적**: Late Event 상황을 재현하여 Processing Time 방식의 한계 체감
- **방법**: 전체 이벤트의 20%를 5초 지연시켜 전송
- **구현**: `heapq`(최소 힙) 기반 지연 전송 큐

---
## 🔬 Chaos 구현: heapq 기반 지연 전송

Generator는 `heapq`를 사용하여 Late Event를 구현합니다.

### 📚 heapq란?

**heapq**: Python 표준 라이브러리의 **최소 힙(min-heap)** 자료구조

| 특징 | 설명 |
|------|------|
| **자동 정렬** | 리스트를 힙으로 관리하면 항상 가장 작은 값이 `[0]`번 인덱스에 위치 |
| **시간 복잡도** | `heappush`: O(log n), `heappop`: O(log n) |
| **사용 사례** | 우선순위 큐, 스케줄링, Top-K 알고리즘 등 |

### Stream Lab에서의 활용

```python
scheduled = []  # (전송_예정_시각, event) 튜플을 저장
```

- 튜플 비교는 첫 번째 원소(전송_예정_시각)부터 하므로
- `scheduled[0]`에는 항상 **"가장 빨리 보내야 할 이벤트"**가 자동으로 위치

In [ ]:
# === src/producer.py: heapq 초기화 ===

import heapq

# 지연 전송 예약 큐
scheduled = []  # (send_time, event) 튜플을 저장

# 핵심 연산:
#   heapq.heappush(scheduled, (send_time, event))  # 새 예약 추가 O(log n)
#   heapq.heappop(scheduled)                       # 가장 빠른 예약 꺼내기 O(log n)

### 1️⃣ Late Event 예약 (heappush)

In [ ]:
# === Late Event 예약 로직 ===

# Chaos 설정
CHAOS_ENABLED = True      # Phase 2에서 활성화
CHAOS_LATE_RATE = 0.2     # 20% 확률로 지연
CHAOS_DELAY_SECONDS = 5   # 5초 지연

# 이벤트 생성
event = {
    "user_id": f"U{random.randint(1, 5)}",
    "amount": random.randint(500, 5000),
    "event_time": datetime.now().isoformat(timespec="milliseconds"),
}

# 20% 확률로 5초 지연 예약
if CHAOS_ENABLED and random.random() < CHAOS_LATE_RATE:
    send_time = time.time() + CHAOS_DELAY_SECONDS  # 현재 시각 + 5초
    heapq.heappush(scheduled, (send_time, event))  # 힙에 삽입 (자동 정렬)
    print(
        f"[LATE] 예약됨: {json.dumps(event, ensure_ascii=False)} "
        f"→ {CHAOS_DELAY_SECONDS:.1f}초 후 전송"
    )
else:
    send_event("[NORMAL]", event)  # 즉시 전송

# 💡 핵심:
#   - event_time은 "현재 시각" (생성 시점)
#   - send_time은 "5초 후" (전송 시점)
#   - Late Event의 event_time은 실제 전송보다 5초 빠름!

### 2️⃣ 예약된 이벤트 전송 (heappop)

In [ ]:
# === 메인 루프: 예약 큐 처리 ===

while True:
    now = time.time()

    # 1. 예약 큐에서 전송 시각이 된 이벤트를 꺼내서 전송
    while scheduled:
        earliest_send_time, _earliest_event = scheduled[0]  # peek: 꺼내지 않고 확인
        if earliest_send_time > now:
            break  # 아직 전송 시각이 안 됐으면 탈출

        _send_time, event = heapq.heappop(scheduled)  # pop: 실제로 큐에서 제거
        send_event("[LATE]", event)

    # 2. 새 이벤트 생성 (위의 Late Event 예약 로직)
    # ...

    # 3. 1초 대기
    time.sleep(EVENT_INTERVAL_SECONDS)

# 💡 핵심:
#   - scheduled[0]: 항상 가장 빨리 보내야 할 이벤트 (최소 힙 특성)
#   - peek 후 비교: 시간이 안 됐으면 break (효율적)
#   - pop 후 전송: 시간이 됐으면 큐에서 제거 후 전송

### 📊 heapq 동작 시각화

```
Time: 01:23:05
================
Event 생성: {user_id: "U2", amount: 1500, event_time: "01:23:05"}
→ 20% 확률 hit! → send_time = 01:23:10으로 예약

heappush(scheduled, (01:23:10, event))

scheduled = [
    (01:23:08, {...}),  # 가장 빨리 보낼 것 (scheduled[0])
    (01:23:10, {...}),  # 방금 추가된 것
    (01:23:12, {...}),
]

Time: 01:23:08
================
scheduled[0] = (01:23:08, {...})
earliest_send_time = 01:23:08 <= now = 01:23:08  ✓
→ heappop(scheduled) → Kafka 전송 "[LATE]"

scheduled = [
    (01:23:10, {...}),  # 이제 이게 scheduled[0]
    (01:23:12, {...}),
]
```

---
## ✅ Phase 2 실행

In [ ]:
# === Just 사용 시 ===
just phase2

# === 직접 명령어 실행 시 ===
# # Chaos 모드로 Generator + Consumer 재시작
CHAOS_ENABLED=true RESULT_TABLE=naive_results KAFKA_GROUP_ID=python-consumer-limit \
  docker compose up -d generator python-consumer

### 🔍 환경 변수 설명

| 환경 변수 | 값 | 의미 |
|----------|-------|------|
| `CHAOS_ENABLED` | `true` | Late Event 주입 활성화 |
| `RESULT_TABLE` | `naive_results` | Phase 1과 구분되는 테이블에 저장 |
| `KAFKA_GROUP_ID` | `python-consumer-limit` | 새로운 Consumer Group (offset 리셋) |

In [ ]:
# === 로그에서 Late Event 발생 확인 ===
docker compose logs -f generator | grep LATE

### 📋 예상 로그 (Generator)

```
[LATE] 예약됨: {"user_id": "U3", "amount": 2341, "event_time": "2026-02-09T01:23:07.234"} → 5.0초 후 전송
[NORMAL] 전송됨: {"user_id": "U1", "amount": 1823, "event_time": "2026-02-09T01:23:08.456"}
[NORMAL] 전송됨: {"user_id": "U2", "amount": 3142, "event_time": "2026-02-09T01:23:09.678"}
[LATE] 전송됨: {"user_id": "U3", "amount": 2341, "event_time": "2026-02-09T01:23:07.234"}
```

관찰 포인트:
- `[LATE] 예약됨`: 5초 후 전송 예약 (heappush)
- `[LATE] 전송됨`: 5초 후 실제 전송 (heappop)
- event_time은 예약 시점 (01:23:07), 전송은 5초 후 (01:23:12)

In [ ]:
# === Late Event 오배치 확인 (Consumer 로그) ===
docker compose logs python-consumer | grep "LATE-DETECT"

### 📋 예상 로그 (Consumer - Late Event 감지)

```
[PROC] window_start=2026-02-09 01:23:10 user=U3 amount=2341 sum=2341 event_time=2026-02-09T01:23:07.234 proc_time=2026-02-09T01:23:12.456
  └─ [LATE-DETECT] 오배치! event_time(2026-02-09T01:23:07.234) → 윈도우 2026-02-09 01:23:00 에 속해야 하지만, 현재 윈도우 2026-02-09 01:23:10 에 집계됨
```

**문제 발생!**
- event_time: 01:23:07 → 원래 윈도우 01:23:00~01:23:10
- proc_time: 01:23:12 → 현재 윈도우 01:23:10~01:23:20
- **윈도우 오배치**: 2341원이 잘못된 윈도우에 집계됨

### 🔍 Late Event 감지 코드

Consumer는 모든 이벤트의 event_time을 체크하여 오배치를 감지합니다.

In [ ]:
# === src/naive_consumer.py: Late Event 감지 ===

# event_time이 있으면 올바른 윈도우 계산
if event_time:
    correct_window = floor_window_start(datetime.fromisoformat(event_time))

    # 현재 윈도우와 다르면 Late Event!
    if correct_window != current_window_start:
        print(
            f"  └─ [LATE-DETECT] 오배치! "
            f"event_time({event_time}) → 윈도우 {correct_window} 에 속해야 하지만, "
            f"현재 윈도우 {current_window_start} 에 집계됨"
        )

    # 모든 이벤트를 event_log 테이블에 기록 (비교 분석용)
    with conn.cursor() as cur:
        cur.execute(
            "INSERT INTO event_log "
            "(event_time, proc_time, assigned_window, correct_window, user_id, amount) "
            "VALUES (%s, %s, %s, %s, %s, %s)",
            (event_time, datetime.now(), current_window_start, correct_window, user_id, amount),
        )
    conn.commit()

# 💡 핵심:
#   - assigned_window: Processing Time 기준 (실제 집계된 윈도우)
#   - correct_window: Event Time 기준 (원래 속해야 할 윈도우)
#   - 두 값이 다르면 Late Event!

---
## 📊 Phase 2 결과 비교: "총합계는 같은데 윈도우 분포가 다르다"

In [ ]:
# === 1분 대기 후 PostgreSQL 접속 ===
sleep 60
docker compose exec postgres psql -U postgres -d streamdb

In [ ]:
# === 쿼리 1: 전체 합계 비교 (동일해야 함) ===
SELECT
    'Processing Time 기준' AS method, SUM(amount) AS grand_total
FROM event_log
UNION ALL
SELECT
    'Event Time 기준', SUM(amount)
FROM event_log;

### 📋 예상 결과 (전체 합계)

```
        method        | grand_total
----------------------+-------------
 Processing Time 기준 |      216567
 Event Time 기준      |      216567   ← 동일!
```

**관찰 포인트:**
- 두 방식 모두 **같은 이벤트**를 집계했으므로 총합계는 동일
- 문제는 "윈도우별 분포"에서 발생!

In [ ]:
# === 쿼리 2: 윈도우별 비교 (여기서 차이 드러남!) ===
SELECT
    COALESCE(p.window_start, e.window_start) AS window_start,
    COALESCE(p.proc_total, 0) AS proc_total,
    COALESCE(e.event_total, 0) AS event_total,
    COALESCE(p.proc_total, 0) - COALESCE(e.event_total, 0) AS diff
FROM (
    SELECT assigned_window AS window_start, SUM(amount) AS proc_total
    FROM event_log GROUP BY assigned_window
) p
FULL OUTER JOIN (
    SELECT correct_window AS window_start, SUM(amount) AS event_total
    FROM event_log GROUP BY correct_window
) e ON p.window_start = e.window_start
ORDER BY window_start;

### 📋 예상 결과 (윈도우별 비교)

```
    window_start     | proc_total | event_total |  diff
---------------------+------------+-------------+--------
 2026-02-09 01:23:00 |      28603 |       28603 |      0  ← 정확
 2026-02-09 01:23:10 |      32522 |       28610 |   3912  ← Late Event 유입 (과대)
 2026-02-09 01:23:20 |      25518 |       21606 |   3912  ← Late Event 유입 (과대)
 2026-02-09 01:23:30 |      22669 |       28931 |  -6262  ← Late Event 빠짐 (과소)
 2026-02-09 01:23:40 |      28109 |       28554 |   -445  ← Late Event 빠짐 (과소)
 2026-02-09 01:23:50 |      27589 |       28136 |   -547  ← Late Event 빠짐 (과소)
```

**해석:**

| diff | 의미 | 원인 |
|------|------|------|
| `diff > 0` | 과대 집계 | Late Event가 잘못 유입됨 |
| `diff < 0` | 과소 집계 | Late Event가 빠져나감 |
| `diff = 0` | 정확 | Late Event 영향 없음 |

### 📋 Late Event 발생/감지 로그 조회

In [ ]:
# === Just 사용 시 ===
# # \q (PostgreSQL 종료)
just late-events

# === 직접 명령어 실행 시 ===
# # \q (PostgreSQL 종료)
docker compose logs generator | grep -E "\[LATE\].*예약됨"
docker compose logs python-consumer | grep "LATE-DETECT"

---
## 📈 타임라인으로 문제 시각화

![](https://cdn.discordapp.com/attachments/1457516082071081045/1470655559752614045/image.png?ex=698f61f8&is=698e1078&hm=a605bc6dbf388ef033a389e2f74bb86fb727636773661d30a0e01ee6b1c5476c&)

---
## 🚨 Phase 2에서 드러난 세 가지 문제

### 1️⃣ 윈도우 오배치 (Window Misalignment)

**문제:**
- Late Event가 원래 속한 윈도우가 아닌 다른 윈도우에 집계됨
- 윈도우별 집계 결과가 부정확해짐

**원인:**
- Processing Time 기준으로 윈도우를 닫기 때문
- event_time을 무시하고 현재 윈도우에 무조건 누적

**영향:**
- 실시간 모니터링 대시보드의 그래프가 부정확함
- "지난 10초 동안 결제 금액" 같은 지표를 신뢰할 수 없음

### 2️⃣ 상태 유실 (State Loss)

**문제:**
- `window_sums` dict는 메모리에만 존재
- 프로세스 종료 시 아직 flush 안 된 집계는 사라짐

**코드:**
```python
finally:
    # 교육 목적상 종료 시 메모리 상태를 강제 flush하지 않음
    consumer.close()
    conn.close()
```

**영향:**
- 컨테이너 재시작, 배포, 장애 시 데이터 유실
- 운영 환경에서는 치명적!

### 3️⃣ INSERT Only (UPDATE 불가)

**문제:**
- 윈도우 닫힐 때 DB에 INSERT만 가능
- 늦게 도착한 데이터로 이미 닫힌 윈도우를 UPDATE할 수 없음

**코드:**
```python
query = (
    f"INSERT INTO {table_name} (window_start, user_id, total_amount) "
    "VALUES (%s, %s, %s)"
)
```

**영향:**
- Late Event 처리 불가 (이미 flush된 윈도우는 수정 불가)
- Event Time 기준 집계를 구현하려면 UPSERT 필요

---
## 🔍 Phase 2 관찰 포인트

| 항목 | 관찰 내용 |
|------|----------|
| **Late Event 발생** | 전체 이벤트의 약 20%가 5초 지연 |
| **오배치 감지** | `[LATE-DETECT]` 로그로 확인 가능 |
| **전체 합계** | Processing Time ≈ Event Time (같은 이벤트) |
| **윈도우별 차이** | `diff ≠ 0`인 윈도우 다수 발견 |
| **과대/과소 집계** | Late Event 유입/빠짐으로 인한 부정확성 |

---
## ✅ Phase 2 정리

In [ ]:
# === Just 사용 시 ===
just phase2-done

# === 직접 명령어 실행 시 ===
docker compose stop generator python-consumer

**Phase 2에서 배운 것:**
- ❌ Processing Time만으로는 Late Event를 정확히 처리할 수 없다
- ❌ 메모리 기반 상태는 프로세스 종료 시 유실된다
- ❌ INSERT Only 방식은 이미 닫힌 윈도우를 수정할 수 없다

**근본 원인:**
- event_time을 무시하고 proc_time으로만 윈도우를 관리
- 윈도우가 닫히면 다시 열 수 없음 (재계산 불가)

**필요한 해결책:**
- ✅ Event Time 기준 윈도우 (이벤트 발생 시각 기준)
- ✅ Watermark (윈도우를 언제 닫을지 결정)
- ✅ Checkpointing (상태를 영구 저장)
- ✅ UPSERT (이미 닫힌 윈도우도 UPDATE 가능)

→ **Apache Flink**가 이 모든 문제를 해결합니다! (Phase 4에서 확인)

---
---
# 📝 FAQ

**Q1. Chaos Engineering을 실무에서 정말 사용하나요?**

네! Netflix, Amazon, Google 등 대형 기업에서 프로덕션 환경에 의도적으로 장애를 주입하여
시스템의 복원력을 검증합니다. 주요 도구:
- **Chaos Monkey**: 랜덤하게 서버 종료
- **Chaos Kong**: 전체 AWS 리전 장애 시뮬레이션
- **Chaos Mesh**: Kubernetes 환경용 Chaos 플랫폼

Stream Lab에서는 "Late Event 재현"이라는 제한된 범위로 Chaos 개념을 적용했습니다.

**Q2. heapq 말고 다른 자료구조를 써도 되나요?**

예약된 이벤트를 "전송 시각 순서"로 관리해야 하므로 우선순위 큐가 필요합니다.
- `list.sort()`: 매번 정렬하면 O(n log n) (비효율적)
- `heapq`: 삽입/삭제 O(log n) (효율적)
- `queue.PriorityQueue`: heapq 기반 thread-safe 래퍼 (단일 스레드에서는 오버헤드)

heapq가 가장 간단하고 효율적인 선택입니다.

**Q3. Processing Time vs Event Time, 실무에서 어떤 문제가 더 심각한가요?**

**Event Time 무시로 인한 실제 사고 사례:**
- 금융 사기 탐지: 5초 전 발생한 이상 거래를 놓쳐 수억원 손실
- IoT 센서: 네트워크 복구 후 과거 데이터가 몰려오면서 알람 폭주
- 광고 과금: 클릭 이벤트 시간이 틀려 광고주에게 과다 청구 → 소송

**Processing Time이 적합한 경우:**
- "지금 이 순간 서버 CPU 사용률" 같은 실시간 모니터링
- Event Time이 없거나 중요하지 않은 로그 수집

**Q4. Late Event 비율 20%는 현실적인가요?**

**실무 환경별 Late Event 비율:**
- 모바일 앱 (4G/5G): 1~5% (네트워크 품질 양호)
- 모바일 앱 (지하철/터널): 10~30% (네트워크 단절 빈번)
- IoT 센서: 5~15% (배터리 절약 모드, 네트워크 불안정)
- 금융 거래: 0.1~1% (안정적 인프라)

20%는 **최악의 시나리오**를 가정한 것으로, 교육 목적상 문제를 명확히 드러내기 위한 설정입니다.

**Q5. Python으로는 이 문제를 해결할 수 없나요?**

이론적으로는 가능하지만:
- Event Time 기준 윈도우 관리 (복잡한 로직)
- Watermark 구현 (수동으로 타이머 관리)
- Checkpointing (Redis/RocksDB 연동)
- UPSERT 쿼리 (DB 부하 증가)

결과적으로 **Flink 수준의 기능을 직접 구현하게 됨** (재발명의 수레바퀴).
이미 검증된 도구(Flink)를 사용하는 것이 현명한 선택입니다.

---
# ✅ 퀴즈

**Q1.** Late Event의 정의로 가장 정확한 것은?

- A) 네트워크 지연으로 발생한 모든 이벤트
- B) Event Time과 Processing Time이 다른 모든 이벤트
- C) 원래 속한 윈도우가 이미 닫힌 후에 도착하는 이벤트
- D) 5초 이상 지연된 이벤트

<details>
<summary>정답 확인</summary>

**정답: C)**

Late Event는 단순히 "지연된 이벤트"가 아니라 **"윈도우 경계를 넘어 도착한 이벤트"**입니다.
Event Time이 10:00:05인데 Processing Time 10:00:12에 도착했다면,
10:00:00~10:00:10 윈도우가 이미 닫혔으므로 Late Event입니다.

</details>

---

**Q2.** heapq의 특징으로 올바른 것은?

- A) 항상 가장 큰 값이 [0]번 인덱스에 위치한다
- B) 삽입/삭제의 시간 복잡도가 O(1)이다
- C) 튜플을 저장하면 첫 번째 원소 기준으로 자동 정렬된다
- D) thread-safe하다

<details>
<summary>정답 확인</summary>

**정답: C)**

heapq는 **최소 힙(min-heap)**으로, 가장 **작은** 값이 [0]번 인덱스에 위치합니다 (A는 틀림).
시간 복잡도는 삽입/삭제 모두 **O(log n)**입니다 (B는 틀림).
Python의 heapq는 thread-safe하지 않습니다 (D는 틀림).

튜플 `(send_time, event)`를 저장하면 `send_time` 기준으로 자동 정렬되므로 **C가 정답**입니다.

</details>

---

**Q3.** Phase 2에서 "전체 합계는 같은데 윈도우별 분포가 다른" 이유로 가장 정확한 것은?

- A) Kafka 메시지가 유실되었다
- B) Late Event가 원래 윈도우가 아닌 다른 윈도우에 집계되었다
- C) Processing Time이 Event Time보다 항상 빠르다
- D) heapq가 잘못 정렬했다

<details>
<summary>정답 확인</summary>

**정답: B)**

전체 합계가 동일하다는 것은 **같은 이벤트**를 집계했다는 의미입니다 (A 유실 없음).
하지만 Late Event가 event_time 기준 윈도우(01:23:00~01:23:10)가 아닌
proc_time 기준 윈도우(01:23:10~01:23:20)에 집계되면서 **윈도우별 분포가 달라졌습니다**.

Processing Time이 Event Time보다 빠를 수는 없으며 (C는 틀림),
heapq는 정상 동작합니다 (D는 틀림).

</details>

---
# 📌 핵심 요약

| 개념 | 핵심 내용 |
|------|----------|
| **Phase 1 목표** | Python만으로도 실시간 집계가 가능함을 확인 |
| **Processing Time** | `datetime.now()` 기준 윈도우 (이벤트 도착 시각) |
| **window_sums** | 메모리(dict) 기반 상태, 10초마다 DB flush 후 초기화 |
| **Phase 1 결과** | Chaos OFF 상태에서는 정확한 집계 가능 |
| **Late Event 정의** | 원래 속한 윈도우가 이미 닫힌 후에 도착하는 이벤트 |
| **Late Event 원인** | 네트워크 불안정, 기기 버퍼링, 시스템 부하, 분산 환경 |
| **Chaos Engineering** | 의도적 장애 주입으로 시스템 복원력 검증 |
| **heapq** | 최소 힙, (send_time, event) 튜플로 예약 큐 관리 |
| **Phase 2 결과** | 전체 합계는 동일, 윈도우별 분포는 부정확 |
| **세 가지 문제** | ① 윈도우 오배치 ② 상태 유실 ③ INSERT Only |
| **근본 원인** | event_time 무시, Processing Time만으로 윈도우 관리 |
| **필요한 해결책** | Event Time, Watermark, Checkpointing, UPSERT |
| **다음 단계** | Phase 3 (이론), Phase 4 (Flink 구현) |


% [markdown]
---
---
## 다음 파일

# ⚡ 2교시: Stream Lab - Phase 3 & Phase 4

## 🎯 학습 목표
- Phase 2에서 발견한 문제를 해결하기 위한 4가지 핵심 요구사항 도출
- Event Time, Watermark, 상태 관리, UPSERT 개념 이해
- Flink가 4가지 문제를 어떻게 해결하는지 실습
- Chaos 모드에서도 정확한 결과를 내는 Flink Job 실행 및 검증

---
## Phase 3: "그러면 뭐가 필요할까?"

Phase 2에서 우리는 Late Event 환경에서 Python Consumer가 부정확한 결과를 낸다는 것을 확인했습니다.

이제 **"정확한 실시간 처리"를 위해 무엇이 필요한지** 생각해봅시다.

### 생각해보기: 4가지 질문

Phase 2의 문제를 해결하려면 다음 질문들에 답해야 합니다.

#### 질문 1: 시간 기준

**Q**: Late Event 문제를 해결하려면 어떤 시간을 기준으로 윈도우를 나눠야 할까?

**두 가지 선택지:**
- **Processing Time(처리 시간)**: Consumer가 메시지를 받은 시간
  - Python 코드: `datetime.now()`
- **Event Time(이벤트 시간)**: 실제 결제가 발생한 시간
  - Python 코드: `row["event_time"]` (데이터 내부 필드)

**어느 것이 "진실"인가?**

In [ ]:
# === Processing Time vs Event Time ===

# 📌 Processing Time (Python Consumer의 현재 방식)
#
#   실제 발생 시간: 10:00:05
#   네트워크 지연으로 도착: 10:00:12
#
#   Python Consumer가 본 시간: 10:00:12 ← Processing Time
#   → 10:00:10 ~ 10:00:20 윈도우에 배치 (잘못됨!)
#
#
# 📌 Event Time (올바른 방식)
#
#   실제 발생 시간: 10:00:05 ← Event Time
#   네트워크 지연으로 도착: 10:00:12
#
#   데이터 내부의 event_time 필드를 확인: 10:00:05
#   → 10:00:00 ~ 10:00:10 윈도우에 배치 (정확함!)
#
#
# 💡 결론: Event Time을 기준으로 윈도우를 나눠야 정확한 집계가 가능하다.

#### 질문 2: 늦은 데이터 대기

**Q**: Event Time 기준으로 윈도우를 나눈다면, 언제까지 Late Event를 기다려야 할까?

**딜레마:**
- 무한정 기다릴 수 없음 → 윈도우를 닫아야 결과 출력 가능
- 너무 빨리 닫으면 → Late Event가 누락됨

**해결책: Watermark**
- "5초까지는 기다리자" 같은 기준 설정
- Watermark = Event Time - 허용 지연 시간

In [ ]:
# === Watermark 개념 ===

# 📌 Watermark란?
#   "이 시점까지의 이벤트는 다 왔다"고 판단하는 기준선
#
#   예: Watermark = event_time - 5초
#
# ┌─────────────────────────────────────────────────────┐
# │           Window: 10:00:00 ~ 10:00:10               │
# ├─────────────────────────────────────────────────────┤
# │                                                     │
# │  ✅ event_time=10:00:03, 도착=10:00:08               │
# │     → Watermark=10:00:03 → 윈도우에 포함               │
# │                                                     │
# │  ✅ event_time=10:00:09, 도착=10:00:15               │
# │     → Watermark=10:00:10 → 윈도우 닫히기 직전 포함       │
# │                                                     │
# │  ❌ event_time=10:00:02, 도착=10:00:16               │
# │     → Watermark=10:00:11 > 10:00:10 → 너무 늦음       │
# │                                                     │
# └─────────────────────────────────────────────────────┘
#
# Watermark가 윈도우 끝(10:00:10)을 넘으면 윈도우 닫힘 → 결과 출력
#
#
# 💡 Watermark 설정 가이드라인:
#   - 짧게 (예: 1초): 결과 빨리 나오지만 Late Event 누락 위험
#   - 길게 (예: 1분): Late Event 대부분 포함하지만 결과 지연
#   - 실전: 데이터 특성 분석 후 결정 (P95/P99 지연 시간 기준)

#### 질문 3: 상태 관리

**Q**: Python Consumer가 재시작되면 집계 중인 데이터가 사라진다. 어떻게 해결할까?

**현재 Python 구현의 문제:**

In [ ]:
# === Python Consumer의 상태 관리 (src/naive_consumer.py:99) ===

# window_sums = {}   # {user_id: total_amount} - 메모리에만 존재!

# 문제:
# 1. 프로세스 종료 → dict 소멸 → 집계 중인 데이터 유실
# 2. 아직 flush되지 않은 윈도우 집계도 함께 사라짐
# 3. RAM 크기에 제한됨 (대규모 상태 관리 불가)
#
#
# 필요한 것:
# - 디스크에 상태 저장 (예: RocksDB)
# - 주기적으로 체크포인트 생성 → 재시작 시 복구
# - 대용량 상태 관리 지원 (수백 GB ~ PB급)

**RocksDB란?**

Meta(Facebook)에서 개발한 고성능 임베디드 Key-Value 저장소
- SSD 환경에 최적화되어 빠른 읽기/쓰기 지원
- Flink나 Kafka Streams의 기본 상태 저장소로 활용
- 메모리가 아닌 디스크에 저장하므로 대용량 상태 관리 가능

#### 질문 4: 결과 업데이트

**Q**: Late Event가 도착했을 때, 이미 DB에 저장된 윈도우 결과를 어떻게 수정할까?

**현재 Python 구현의 DB 쓰기 방식:**

In [ ]:
# === Python Consumer의 DB 쓰기 (src/naive_consumer.py:60-66) ===

# query = (
#     f"INSERT INTO {table_name} (window_start, user_id, total_amount) "
#     "VALUES (%s, %s, %s)"
# )
# with conn.cursor() as cur:
#     cur.executemany(query, rows)

# → 항상 INSERT만 수행
# → 같은 윈도우/유저 결과가 다시 생성되면 중복 행이 발생

**DB 스키마 비교:**

In [ ]:
# === baseline/naive 테이블 vs flink_results 테이블 ===

# -- sql/init.sql: baseline/naive 테이블에는 PK가 없다
CREATE TABLE baseline_results (
    window_start TIMESTAMP,
    user_id      VARCHAR(50),
    total_amount INT,
    updated_at   TIMESTAMP DEFAULT NOW()
);
#
CREATE TABLE naive_results (
    window_start TIMESTAMP,
    user_id      VARCHAR(50),
    total_amount INT,
    updated_at   TIMESTAMP DEFAULT NOW()
);
#
#
# -- flink_results 테이블에는 PK가 있다 → UPSERT 가능
CREATE TABLE flink_results (
    window_start TIMESTAMP,
    window_end   TIMESTAMP,
    user_id      VARCHAR(50),
    total_amount INT,
    updated_at   TIMESTAMP DEFAULT NOW(),
    PRIMARY KEY (window_start, window_end, user_id)
);
#
#
# 필요한 것:
# - UPSERT (있으면 UPDATE, 없으면 INSERT)
# - Primary Key 설정 (window_start, window_end, user_id)

### 요구사항 정리

**생각해보기**를 통해 도출된 요구사항:

| 요구사항 | Python 구현 | 필요한 기능 |
|---------|------------|------------|
| **정확한 윈도우 분류** | Processing Time (부정확) | Event Time 기반 처리 |
| **Late Event 대응** | 윈도우 닫히면 끝 | Watermark (지연 허용) |
| **상태 복구** | dict (휘발성) | RocksDB + Checkpointing |
| **결과 수정** | INSERT만 가능 | UPSERT (Primary Key 기반) |

**결론**: 이런 기능들을 직접 구현하는 것은 매우 복잡하다.

→ **Apache Flink 같은 전문 스트림 처리 프레임워크가 필요!**

---
## Phase 4: "Flink는 이걸 어떻게 해결하는가"

### 목표

PyFlink로 동일한 집계를 구현하고, Chaos 모드에서도 정확한 결과를 내는 것을 확인합니다.

### Flink 아키텍처

Flink는 **JobManager**와 **TaskManager**로 구성된 클러스터입니다.

In [ ]:
# === Flink 클러스터 구조 ===

# ┌────────────────────────────────────────────────────────────┐
# │                    Flink Cluster                           │
# ├────────────────────────────────────────────────────────────┤
# │                                                            │
# │  [JobManager]                 [TaskManager]                │
# │  - Job 관리                   - Task 실행                    │
# │  - Checkpoint 조율            - RocksDB 상태 저장             │
# │  - Web UI (8081)              - Kafka Consumer             │
# │                               - Window 집계                 │
# │                               - PostgreSQL Writer          │
# │                                                            │
# │  Checkpoint Storage: /tmp/flink-checkpoints (컨테이너 내부)   │
# │  State Backend: RocksDB (디스크 기반, 복구 가능)                │
# └────────────────────────────────────────────────────────────┘
#
#
# 📌 JobManager:
#   - Job의 전체 흐름을 관리하는 마스터 노드
#   - Checkpoint 조율 및 장애 복구
#   - Web UI 제공 (http://localhost:8081)
#
# 📌 TaskManager:
#   - 실제 데이터 처리를 수행하는 워커 노드
#   - RocksDB로 상태를 디스크에 저장
#   - 여러 개의 Task Slot을 가지고 병렬 처리

### 실행 단계

> **`just phase4` 한 줄이면 아래 1~4단계를 자동으로 수행합니다.**
>
> 수동으로 실행하려면 아래 순서를 따르세요.

#### Step 1: Flink 클러스터 시작

In [ ]:
# === 1단계: Flink 클러스터 시작 ===

# 기존 서비스 중지 & 테이블 초기화
docker compose stop generator python-consumer
docker compose exec postgres psql -U postgres -d streamdb \
  -c "TRUNCATE naive_results; TRUNCATE event_log;"

# JobManager + TaskManager 시작
docker compose up -d jobmanager taskmanager

# 클러스터 상태 확인
docker compose logs jobmanager | grep "started"

# Web UI 접속 (브라우저)
# http://localhost:8081

#### Step 2: Kafka 토픽 생성

**중요**: Flink Job을 제출하기 전에 Kafka 토픽을 먼저 생성해야 합니다.
토픽이 없으면 Flink Job이 `UnknownTopicOrPartitionException`으로 재시작 루프에 빠집니다.

In [ ]:
# === 2단계: Kafka 토픽 생성 ===

# Kafka 토픽 미리 생성 (Flink Job이 토픽 없음 에러를 내지 않도록)
docker compose exec kafka /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create --topic payment-log \
  --partitions 1 --replication-factor 1 \
  --if-not-exists

# 토픽 생성 확인
docker compose exec kafka /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --list

#### Step 3: Flink Job 제출

**⚠️ 주의: Flink Job을 Generator보다 먼저 제출해야 합니다!**

Flink의 Kafka Source는 `scan.startup.mode = latest-offset`로 설정되어 있어,
Job 제출 시점 이후에 Kafka에 들어온 이벤트만 읽습니다.

Generator가 먼저 시작되면 그 사이 생산된 이벤트를 Flink가 놓쳐서
첫 몇 개 윈도우에서 Flink 결과가 정답보다 적게 나옵니다.

In [ ]:
# === 3단계: Flink Job 제출 ===

# Flink 클러스터 준비 대기 (약 40초)
docker compose exec jobmanager flink list
# "No running jobs." 확인

# PyFlink Job 실행 (Generator보다 먼저! 첫 이벤트부터 수신)
docker compose exec jobmanager flink run -py /opt/flink/src/flink_job.py

# Job 실행 확인 (Web UI에서 Running Jobs 확인 또는)
docker compose exec jobmanager flink list

#### Step 4: Chaos 모드로 Generator + Consumer 동시 시작

이제 Generator와 Python Consumer를 동시에 시작하여
Late Event가 발생하는 환경을 만듭니다.

In [ ]:
# === 4단계: Chaos 모드로 Generator + Consumer 동시 시작 ===

# Generator + Consumer를 동시에 시작 (같은 시점부터 이벤트 생산/소비)
CHAOS_ENABLED=true RESULT_TABLE=naive_results KAFKA_GROUP_ID=python-consumer-flink \
  docker compose up -d generator python-consumer

# Late Event 발생 확인
docker compose logs -f generator | grep LATE

# TaskManager 로그 확인
docker compose logs -f taskmanager

### 실시간 처리 확인

1분 후부터 PostgreSQL에서 Flink 결과를 조회할 수 있습니다.

In [ ]:
# === PostgreSQL에서 Flink 결과 조회 ===

docker compose exec postgres psql -U postgres -d streamdb

# -- Flink 결과 확인
SELECT
    window_start,
    window_end,
    user_id,
    total_amount,
    updated_at
FROM flink_results
ORDER BY window_start, user_id
LIMIT 20;
#
#
# -- 특정 윈도우의 업데이트 횟수 확인 (UPSERT 동작)
SELECT
    window_start,
    user_id,
    total_amount,
    updated_at
FROM flink_results
WHERE window_start = (SELECT MIN(window_start) FROM flink_results)
ORDER BY user_id;

### 최종 비교: Flink vs Python vs 정답 (3분 실행 후)

Phase 4에서는 Generator + Python Consumer + Flink가 **같은 Kafka 이벤트**를 동시에 처리합니다.

`event_log` 테이블에 기록된 개별 이벤트의 "올바른 윈도우"와 Flink 결과를 비교할 수 있습니다.

In [ ]:
# === 전체 합계 비교 ===

docker compose exec postgres psql -U postgres -d streamdb

# -- 1) 전체 합계: Flink는 Watermark 지연으로 마지막 윈도우가 미방출되어 약간 적을 수 있음
SELECT 'Flink (Event Time)' AS method, SUM(total_amount) AS grand_total FROM flink_results
UNION ALL
SELECT '정답 (Event Time)', SUM(amount) FROM event_log;

In [ ]:
# === 윈도우별 비교 ===

# -- 2) 윈도우별 비교: Flink는 정답에 가깝고, Python은 틀림
SELECT
    COALESCE(f.window_start, e.window_start) AS window_start,
    COALESCE(f.flink_total, 0)  AS "Flink",
    COALESCE(e.event_total, 0)  AS "정답(Event Time)",
    COALESCE(p.proc_total, 0)   AS "Python(Proc Time)"
FROM (
    SELECT window_start, SUM(total_amount) AS flink_total
    FROM flink_results GROUP BY window_start
) f
FULL OUTER JOIN (
    SELECT correct_window AS window_start, SUM(amount) AS event_total
    FROM event_log GROUP BY correct_window
) e ON f.window_start = e.window_start
FULL OUTER JOIN (
    SELECT assigned_window AS window_start, SUM(amount) AS proc_total
    FROM event_log GROUP BY assigned_window
) p ON COALESCE(f.window_start, e.window_start) = p.window_start
ORDER BY window_start;

### 예상 결과

**1) 전체 합계**: Flink와 event_log의 합계가 근사합니다.

차이가 나는 이유:
- 첫 1~2개 윈도우: Flink와 Consumer의 시작 시점 차이
- 마지막 윈도우: Flink의 Watermark(`event_time - 5초`) 때문에 아직 방출되지 않음

**2) 윈도우별 비교**: 안정 구간에서 **Flink = 정답**, Python은 다릅니다.

In [ ]:
# === 예상 결과 예시 ===

#     window_start     | Flink  | 정답(Event Time) | Python(Proc Time)
# ---------------------+--------+------------------+-------------------
#  2026-02-09 03:51:00 |  34000 |   34000          |  38485  ← naive 과대
#  2026-02-09 03:51:10 |  29033 |   29033          |  24225  ← naive 과소
#  2026-02-09 03:51:20 |  26421 |   26421          |  31229  ← naive 과대
#  2026-02-09 03:52:00 |  27662 |   27662          |  27662  ← Late Event 없는 윈도우
#  2026-02-09 03:52:10 |  29074 |   29074          |  26743  ← naive 과소
#  2026-02-09 03:52:20 |  32888 |   32888          |  40055  ← naive 과대
#
# - Flink = 정답: Event Time + Watermark 덕분에 Late Event도 올바른 윈도우에 배치
# - Python ≠ 정답: Processing Time 기준이므로 Late Event가 잘못된 윈도우에 배치
#
#
# 💡 참고: 첫/마지막 윈도우 차이
#   - 첫 1~2개 윈도우: Flink와 Consumer의 시작 시점 차이로 Flink 값이 더 작을 수 있음
#   - 마지막 윈도우: Flink의 Watermark(`event_time - 5초`) 때문에 아직 방출되지 않아 0으로 표시됨
#   - 이것은 정상 동작입니다. Flink는 Late Event를 기다리기 위해 의도적으로 결과 방출을 지연합니다

---
## Flink가 해결한 4가지

Phase 3에서 도출한 4가지 요구사항을 Flink가 어떻게 해결하는지 코드와 함께 살펴봅시다.

### 1. Event Time 기반 처리

**요구사항**: Processing Time이 아닌 Event Time으로 윈도우를 분류해야 한다.

**Flink 해결책**: Kafka Source DDL에서 `event_time` 필드를 지정하고 Watermark를 설정합니다.

In [ ]:
# === flink_job.py: Kafka Source DDL ===

CREATE TABLE payment_source (
    user_id STRING,
    amount INT,
    event_time TIMESTAMP(3),
    WATERMARK FOR event_time AS event_time - INTERVAL '5' SECOND
) WITH (
    'connector' = 'kafka',
    'topic' = 'payment-log',
    'properties.bootstrap.servers' = 'kafka:9092',
    'properties.group.id' = 'flink-payment-group',
    'scan.startup.mode' = 'latest-offset',
    'format' = 'json',
    'json.timestamp-format.standard' = 'ISO-8601',
    ...
)

# 핵심:
# - event_time TIMESTAMP(3): 데이터 내부의 타임스탬프 필드
# - WATERMARK FOR event_time: 이 필드를 Event Time으로 사용
# - json.timestamp-format.standard = ISO-8601: ISO 형식 파싱

# → event_time 필드 기준으로 윈도우 분류 (도착 시간 무관)

### 2. Watermark (Late Event 대응)

**요구사항**: Late Event를 일정 시간까지 기다려야 한다.

**Flink 해결책**: WATERMARK 선언과 TUMBLE Window를 사용합니다.

In [ ]:
# === flink_job.py: TUMBLE Window 집계 ===

INSERT INTO flink_results
SELECT
    window_start,
    window_end,
    user_id,
    CAST(SUM(amount) AS INT) AS total_amount,
    CAST(CURRENT_TIMESTAMP AS TIMESTAMP(3)) AS updated_at
FROM TABLE(
    TUMBLE(
        TABLE payment_source,
        DESCRIPTOR(event_time),
        INTERVAL '10' SECOND
    )
)
GROUP BY window_start, window_end, user_id

# 핵심:
# - TUMBLE(..., DESCRIPTOR(event_time), INTERVAL '10' SECOND): Event Time 기준 10초 윈도우
# - WATERMARK FOR event_time AS event_time - INTERVAL '5' SECOND: 5초까지 늦은 이벤트 허용

**Watermark 동작 예시:**

In [ ]:
# === Watermark 동작 ===

# 윈도우: 00:00:00 ~ 00:00:10
# Watermark 설정: event_time - 5초
#
# 이벤트 도착 순서:
#   1) event_time=00:00:07 도착 → Watermark = 00:00:02
#      → 00:00:02 < 00:00:10 (윈도우 끝) → 윈도우 아직 열려있음
#
#   2) event_time=00:00:12 도착 → Watermark = 00:00:07
#      → 00:00:07 < 00:00:10 → 윈도우 아직 열려있음
#
#   3) event_time=00:00:16 도착 → Watermark = 00:00:11
#      → 00:00:11 > 00:00:10 (윈도우 끝) → 윈도우 닫힘!
#
# 이 시점에서 event_time 00:00:00~00:00:10 사이의 모든 이벤트가 집계되어 출력됨.
# 5초 늦게 도착한 이벤트도 Watermark 덕분에 올바른 윈도우에 포함됨.
#
#

# 📌 Watermark 흐름 다이어그램:
#
# ![](https://cdn.discordapp.com/attachments/1457516082071081045/1470656326634836040/image.png?ex=698f62af&is=698e112f&hm=1ed9ebcc3e37ba72d15f32da957219344c01ff20a3229cf19e388d53358096fe&)

### 3. RocksDB + Checkpointing (상태 복구)

**요구사항**: 프로세스 재시작 시에도 집계 중인 상태를 복구해야 한다.

**Flink 해결책**: RocksDB State Backend와 Checkpointing을 활성화합니다.

In [ ]:
# === flink_job.py: 상태 관리 설정 (37-58행) ===

# env = StreamExecutionEnvironment.get_execution_environment()
# env.enable_checkpointing(CHECKPOINT_INTERVAL_SECONDS * 1000)  # 10초마다 상태 스냅샷
#
# conf = t_env.get_config().get_configuration()
# conf.set_string("state.backend.type", "rocksdb")                    # 상태를 디스크(RocksDB)에 저장
# conf.set_string("state.checkpoints.dir", CHECKPOINTS_DIR)           # 체크포인트 저장 경로
# conf.set_string("execution.checkpointing.mode", "EXACTLY_ONCE")     # 정확히 한 번 처리 보장
# conf.set_string(
#     "execution.checkpointing.externalized-checkpoint-retention",
#     "RETAIN_ON_CANCELLATION",                                        # 잡 취소해도 체크포인트 보존
# )

# 핵심:
# - state.backend.type = rocksdb: 상태를 메모리가 아닌 디스크에 저장
# - enable_checkpointing(10000): 10초마다 상태 스냅샷 생성
# - EXACTLY_ONCE: 장애 복구 시 정확히 한 번만 처리 보장

**Python Consumer vs Flink 비교:**

In [ ]:
# === 상태 관리 비교 ===

# 📌 Python Consumer:
#   window_sums = {}  # 메모리에만 존재
#
#   프로세스 종료 → dict 소멸 → 집계 중인 데이터 유실
#   RAM 크기에 제한됨
#
#
# 📌 Flink:
#   RocksDB (디스크 기반)
#
#   10초마다 체크포인트로 스냅샷 생성
#   장애 시 마지막 체크포인트에서 복구
#   수백 GB ~ PB급 상태 관리 가능

**상태 복구 테스트:**

In [ ]:
# === TaskManager 재시작 테스트 ===

# TaskManager 강제 종료
docker compose restart taskmanager

# 로그에서 체크포인트 복구 확인
docker compose logs taskmanager | grep "restore"

# Flink는 마지막 체크포인트에서 상태를 복구하여 처리를 이어갑니다.

### 4. UPSERT (결과 수정)

**요구사항**: Late Event 도착 시 이미 저장된 윈도우 결과를 업데이트해야 한다.

**Flink 해결책**: Primary Key를 선언하면 JDBC Connector가 자동으로 UPSERT를 수행합니다.

In [ ]:
# === sql/init.sql: PK 설정으로 Upsert 지원 ===

CREATE TABLE flink_results (
    window_start TIMESTAMP,
    window_end   TIMESTAMP,
    user_id      VARCHAR(50),
    total_amount INT,
    updated_at   TIMESTAMP DEFAULT NOW(),
    PRIMARY KEY (window_start, window_end, user_id)
);

In [ ]:
# === flink_job.py: Sink DDL에서 PK 선언 ===

CREATE TABLE flink_results (
    window_start TIMESTAMP(3),
    window_end TIMESTAMP(3),
    user_id STRING,
    total_amount INT,
    updated_at TIMESTAMP(3),
    PRIMARY KEY (window_start, window_end, user_id) NOT ENFORCED
) WITH (
    'connector' = 'jdbc',
    'url' = 'jdbc:postgresql://postgres:5432/streamdb',
    'table-name' = 'flink_results',
    'username' = 'postgres',
    'password' = 'postgres',
    'driver' = 'org.postgresql.Driver'
)

# 핵심:
# - PRIMARY KEY (window_start, window_end, user_id) NOT ENFORCED: PK 선언
# - Flink JDBC Connector가 PK 기반으로 자동 UPSERT 수행
# - PostgreSQL에서는 ON CONFLICT DO UPDATE로 변환됨

**INSERT vs UPSERT 비교:**

In [ ]:
# === DB 쓰기 전략 비교 ===

# 📌 Python Consumer (INSERT):
INSERT INTO naive_results (window_start, user_id, total_amount)
VALUES ('10:00:00', 'U1', 5000);
#
#   같은 윈도우/유저에 대해 다시 실행하면 → 중복 행 발생
#   Late Event 반영 불가능
#
#
# 📌 Flink (UPSERT):
INSERT INTO flink_results (window_start, window_end, user_id, total_amount)
VALUES ('10:00:00', '10:00:10', 'U1', 5000)
ON CONFLICT (window_start, window_end, user_id)
DO UPDATE SET total_amount = EXCLUDED.total_amount, updated_at = NOW();
#
#   같은 윈도우/유저에 대해 다시 실행하면 → 기존 행 UPDATE
#   Late Event 도착 시 윈도우 결과 자동 수정

---
## Flink Web UI 탐색

Flink Web UI (http://localhost:8081)에서 확인할 것들:

### 1. Running Jobs 탭

- Job Graph 시각화 (Source → Window → Sink)
- Parallelism 설정 확인
- Task 상태 확인 (RUNNING, FINISHED, FAILED)

In [ ]:
# === Web UI - Running Jobs 탭 예시 ===

# ┌─────────────────────────────────────────────────────────┐
# │  Running Jobs                                           │
# ├─────────────────────────────────────────────────────────┤
# │                                                         │
# │  Job: payment-log-aggregation                           │
# │  Status: RUNNING                                        │
# │  Started: 2026-02-10 10:30:15                           │
# │                                                         │
# │  Job Graph:                                             │
# │                                                         │
# │  [Kafka Source]                                         │
# │        │                                                │
# │        ↓                                                │
# │  [TUMBLE Window]                                        │
# │        │                                                │
# │        ↓                                                │
# │  [JDBC Sink]                                            │
# │                                                         │
# │  Parallelism: 1                                         │
# │  Tasks: 3/3 RUNNING                                     │
# │                                                         │
# └─────────────────────────────────────────────────────────┘

### 2. Task Metrics 탭

각 Task의 상세 메트릭을 확인할 수 있습니다.

In [ ]:
# === 주요 메트릭 ===

# 📌 numRecordsIn: Kafka에서 읽은 레코드 수
# 📌 numRecordsOut: PostgreSQL에 쓴 레코드 수
# 📌 currentWatermark: 현재 Watermark 값 (타임스탬프)
# 📌 numLateRecordsDropped: Watermark 이후 도착하여 버려진 레코드 수

# currentWatermark 값을 보면 현재 어느 시점까지의 이벤트를 처리했는지 알 수 있습니다.

### 3. Checkpoints 탭

Checkpoint 성공/실패 이력과 상태 크기를 확인할 수 있습니다.

In [ ]:
# === Web UI - Checkpoints 탭 예시 ===

# ┌─────────────────────────────────────────────────────────┐
# │  Checkpoints                                            │
# ├─────────────────────────────────────────────────────────┤
# │                                                         │
# │  Checkpoint ID  │ Status    │ Duration │ State Size    │
# │  ──────────────────────────────────────────────────────│
# │  12             │ COMPLETED │  523 ms  │  1.2 MB       │
# │  11             │ COMPLETED │  487 ms  │  1.1 MB       │
# │  10             │ COMPLETED │  512 ms  │  1.0 MB       │
# │  ...            │ ...       │  ...     │  ...          │
# │                                                         │
# │  Latest Checkpoint: /tmp/flink-checkpoints/chk-12       │
# │  Checkpoint Interval: 10 seconds                        │
# │  Checkpoint Mode: EXACTLY_ONCE                          │
# │                                                         │
# └─────────────────────────────────────────────────────────┘

# State Size가 계속 증가하면 RocksDB가 상태를 저장하고 있다는 뜻입니다.

---
## 핵심 개념 정리

### 1. Processing Time vs Event Time

| 구분 | Processing Time | Event Time |
|------|----------------|------------|
| **정의** | Consumer가 메시지를 받은 시간 | 이벤트가 실제 발생한 시간 |
| **Python 코드** | `datetime.now()` | `row["event_time"]` |
| **장점** | 구현 간단, 지연 없음 | 정확한 시간 기준 |
| **단점** | Late Event 시 부정확 | Watermark 설정 필요 |
| **사용 사례** | 모니터링, 단순 카운트 | 금융, 분석, SLA |

### 2. 상태 관리 비교

| 구분 | Python dict | Flink RocksDB |
|------|------------|---------------|
| **저장 위치** | 메모리 (RAM) | 디스크 (persistent) |
| **프로세스 재시작** | 데이터 소실 | 체크포인트에서 복구 |
| **크기 제한** | RAM 크기 | 디스크 크기 (수백 GB 가능) |
| **성능** | 빠름 | 약간 느리지만 안정적 |

### 3. DB 쓰기 전략 비교

| 구분 | INSERT | UPSERT |
|------|--------|--------|
| **동작** | 항상 새 행 추가 | 있으면 UPDATE, 없으면 INSERT |
| **중복** | 같은 윈도우 결과 중복 | Primary Key로 유일성 보장 |
| **Late Event 반영** | 불가능 (이미 INSERT됨) | 가능 (기존 행 UPDATE) |
| **SQL** | `INSERT INTO ...` | `INSERT ... ON CONFLICT DO UPDATE` |

### 4. Watermark 전략 가이드라인

| 설정 | 장점 | 단점 | 사용 시나리오 |
|------|------|------|--------------|
| **짧게 (1초)** | 결과 빨리 출력 | Late Event 누락 위험 | 실시간 대시보드 (약간의 오차 허용) |
| **중간 (5초)** | 균형잡힌 선택 | - | 일반적인 실시간 처리 (Stream Lab 기본값) |
| **길게 (1분)** | Late Event 대부분 포함 | 결과 지연 | 정확성이 최우선 (금융, 정산) |

**실전 팁**: 데이터 특성 분석 후 P95/P99 지연 시간을 기준으로 설정

---
## 실습 체크리스트

### Phase 3
- [ ] Event Time vs Processing Time 차이 이해
- [ ] Watermark 개념 이해 (윈도우 닫기 기준)
- [ ] 상태 관리 필요성 이해 (RocksDB + Checkpointing)
- [ ] UPSERT 필요성 이해 (Primary Key 기반)

### Phase 4
- [ ] Flink 클러스터 시작 확인 (Web UI 접속)
- [ ] Kafka 토픽 생성 확인
- [ ] Flink Job 제출 성공 확인 (Generator보다 먼저!)
- [ ] flink_results 테이블에 데이터 UPSERT 확인
- [ ] Chaos 모드에서도 Flink 결과가 정확한 것 확인
- [ ] baseline_results (Chaos OFF)와 flink_results (Chaos ON) 결과 일치 확인
- [ ] Web UI에서 Checkpoint, Watermark 확인

---
## Phase 4 정리

In [ ]:
# === 모든 서비스 중지 ===

# 컨테이너 중지 (데이터는 보존)
docker compose down

# 볼륨까지 삭제 (DB 데이터 초기화)
docker compose down -v

# 이미지까지 삭제 (재빌드 필요 시)
docker compose down --rmi all -v

# 정리 명령어 (Justfile 사용 시)
just clean

**배운 것**:

Flink는 Event Time, Watermark, 상태 관리, UPSERT를 내장으로 제공하여,
Late Event 환경에서도 정확한 실시간 집계를 보장한다.

---
## FAQ

**Q1. Flink 결과의 전체 합계가 정답과 약간 다른데, 왜 그런가요?**

A: 두 가지 이유가 있습니다.

1. **첫 1~2개 윈도우**: Flink와 Generator의 시작 시점이 정확히 일치하지 않아 Flink가 일부 이벤트를 놓칠 수 있습니다.
2. **마지막 윈도우**: Watermark 지연(`event_time - 5초`) 때문에 아직 윈도우가 닫히지 않아 결과가 방출되지 않습니다.

이것은 **정상 동작**입니다. 중간 안정 구간의 윈도우별 비교를 보면 Flink = 정답임을 확인할 수 있습니다.

**Q2. Watermark를 너무 길게 설정하면 안 되나요?**

A: 결과가 지연됩니다.

예를 들어 Watermark를 1분으로 설정하면, 윈도우가 닫히려면 event_time이 윈도우 끝 + 1분이 되어야 합니다.
실시간 대시보드에는 부적합합니다.

반대로 너무 짧게 설정하면 Late Event가 누락됩니다. 데이터 특성에 맞게 조율해야 합니다.

**Q3. Late Event를 완전히 없앨 수는 없나요?**

A: 불가능합니다.

네트워크 지연, GC pause, 시계 불일치 등은 분산 시스템에서 필연적입니다.
대신 Watermark로 "얼마나 기다릴지" 조율하고, Side Output으로 늦은 데이터를 별도 처리할 수 있습니다.

**Q4. Python Consumer를 Flink처럼 구현할 수는 없나요?**

A: 이론적으로는 가능하지만, 매우 복잡합니다.

Event Time 처리, Watermark 생성, 상태 관리, Checkpointing, Exactly-Once, UPSERT 로직을
모두 직접 구현해야 합니다.

Flink는 이런 기능들을 검증된 프레임워크로 제공하므로, 비즈니스 로직에만 집중할 수 있습니다.

**Q5. Flink 대신 Spark Streaming은 안 되나요?**

A: Spark Streaming도 가능하지만, Micro-batch 방식이라 latency가 더 높습니다.

- **Spark Streaming**: 수 초 단위 지연 (Micro-batch)
- **Flink**: 밀리초 단위 지연 (True Streaming, event-by-event)

초저지연이 필요한 금융, 이상 탐지, 실시간 관제 시스템에서는 Flink가 유리합니다.

**Q6. 실제 운영 환경에서는 어떻게 배포하나요?**

A: 주로 다음 두 가지 방식을 사용합니다.

1. **Self-managed**:
   - Kubernetes 위에 Flink Operator 사용
   - YARN, Mesos 같은 클러스터 매니저 활용

2. **Managed Service**:
   - AWS Kinesis Data Analytics for Apache Flink
   - Azure Stream Analytics
   - Confluent Cloud for Apache Flink

운영 부담을 줄이려면 Managed Service가 좋지만, 비용과 제어권을 고려해야 합니다.

---
## 퀴즈

**Q1.** Flink의 Watermark가 `event_time - 5초`로 설정되어 있습니다.
윈도우가 10:00:00 ~ 10:00:10일 때, 이 윈도우가 닫히는 시점은?

- A) 10:00:10에 도착한 이벤트가 들어올 때
- B) 10:00:15의 event_time을 가진 이벤트가 도착할 때
- C) 10:00:10 시각이 되었을 때
- D) 모든 Late Event를 받은 후

<details>
<summary>정답 확인</summary>

**정답: B)**

Watermark = event_time - 5초이므로,
event_time=10:00:15인 이벤트가 도착하면 Watermark=10:00:10이 됩니다.
Watermark가 윈도우 끝(10:00:10)에 도달하면 윈도우가 닫히고 결과가 출력됩니다.

</details>

**Q2.** Flink의 JDBC Connector가 PRIMARY KEY 기반으로 자동 UPSERT를 수행한다고 했습니다.
PostgreSQL에서 실제로 실행되는 SQL은 무엇일까요?

- A) `INSERT INTO ... VALUES ...`
- B) `UPDATE ... SET ... WHERE ...`
- C) `INSERT INTO ... VALUES ... ON CONFLICT DO UPDATE ...`
- D) `MERGE INTO ... USING ...`

<details>
<summary>정답 확인</summary>

**정답: C)**

PostgreSQL의 UPSERT 구문은 `INSERT ... ON CONFLICT DO UPDATE`입니다.
Flink JDBC Connector는 PRIMARY KEY를 감지하여 자동으로 이 구문을 생성합니다.

예:
```sql
INSERT INTO flink_results (window_start, window_end, user_id, total_amount)
VALUES ('10:00:00', '10:00:10', 'U1', 5000)
ON CONFLICT (window_start, window_end, user_id)
DO UPDATE SET total_amount = EXCLUDED.total_amount, updated_at = NOW();
```

</details>

**Q3.** Flink의 RocksDB State Backend를 사용하는 주된 이유는?

- A) 메모리보다 빠른 처리 속도
- B) SQL 쿼리 지원
- C) 대용량 상태 관리 및 장애 복구
- D) GPU 가속 지원

<details>
<summary>정답 확인</summary>

**정답: C)**

RocksDB는 디스크 기반 State Backend로, 다음 이점이 있습니다:
- RAM 크기에 제한되지 않고 수백 GB ~ PB급 상태 관리 가능
- Checkpoint를 통해 장애 복구 시 상태 복원
- 메모리보다는 느리지만, 안정성과 확장성이 뛰어남

메모리 기반 State Backend(Heap)는 속도는 빠르지만 크기 제한이 있고,
프로세스 재시작 시 상태 유실 위험이 있습니다.

</details>

---
## 핵심 요약

### Phase 3 & Phase 4 정리표

| 요구사항 | Python 한계 | Flink 해결책 |
|---------|------------|------------|
| **정확한 윈도우 분류** | Processing Time → Late Event 시 부정확 | Event Time 기반 처리 (`WATERMARK FOR event_time`) |
| **Late Event 대응** | 윈도우 닫히면 끝 | Watermark로 지연 허용 (`event_time - 5초`) |
| **상태 복구** | dict (메모리) → 프로세스 종료 시 유실 | RocksDB + Checkpointing (10초마다 스냅샷) |
| **결과 수정** | INSERT만 → 중복 발생 | UPSERT (PRIMARY KEY 기반 자동 처리) |

### Flink의 4가지 핵심 기능

1. **Event Time**: 데이터 내부 타임스탬프 기준 처리 → 도착 순서 무관
2. **Watermark**: Late Event 허용 시간 설정 → 윈도우 닫기 기준
3. **RocksDB + Checkpointing**: 디스크 기반 상태 저장 → 장애 복구 가능
4. **UPSERT**: PRIMARY KEY 기반 자동 업데이트 → Late Event 반영

### 전체 흐름 다이어그램

# === Stream Lab 전체 흐름 ===

# ![](https://cdn.discordapp.com/attachments/1457516082071081045/1470656722111566000/image.png?ex=698f630e&is=698e118e&hm=19322c64f6c6249a22a7d77e423677af7419028df70a22b4e6a604820fb53761&)

---
## 다음 시간 예고

3교시에서는:
- **Stream Lab Phase 1 & Phase 2 실습** (직접 해보기)
- Chaos OFF 환경에서 Python Consumer 동작 확인
- Chaos ON 환경에서 Late Event 문제 재현
- event_log 테이블로 정답과 비교

지금까지 배운 이론을 직접 실습하며 체화하는 시간을 가지겠습니다!

---
## 📌 마무리

**실시간 데이터 처리는 단순히 "빠르게 처리하는 것"이 아니라,
정확성(Correctness)과 복구성(Fault Tolerance)을 보장하는 것입니다.**

**이것이 Flink 같은 전문 프레임워크가 필요한 이유입니다.**

Phase 3 & Phase 4를 통해 배운 것:
1. Python만으로도 실시간 처리가 가능하지만, 운영 환경에서는 한계가 있다
2. 정확한 처리를 위해서는 Event Time, Watermark, 상태 관리, UPSERT가 필요하다
3. Flink는 이런 기능들을 검증된 프레임워크로 제공한다
4. Chaos 모드에서도 Flink는 정확한 결과를 보장한다


% [markdown]
---
---
## 다음 파일

# 🔍 3교시: Fraud Detection with DataStream API

## 🎯 학습 목표
- Table API vs DataStream API의 차이점 이해
- KeyedStream과 상태 관리(State Management) 개념 학습
- ProcessFunction과 Timer를 활용한 이벤트 패턴 탐지
- 실전 사기 탐지(Fraud Detection) 파이프라인 구현

---
## 1. 왜 DataStream API인가?

1-2교시에서 배운 **Table API**는 SQL 스타일의 선언적 처리에 강력했습니다.
하지만 다음과 같은 복잡한 비즈니스 로직에는 한계가 있습니다.

### 시나리오: 결제 사기 탐지

**요구사항**: "같은 사용자가 소액 결제(< 1000원) 후 1분 이내에 고액 결제(> 10000원)를 하면 사기로 간주"

**왜 이것이 어려운가?**
- 이벤트 간 **시간 관계**(소액 → 1분 → 고액)를 추적해야 함
- 사용자별로 **상태**(직전 소액 결제 기록)를 유지해야 함
- **타이머**로 1분 경과를 감지하여 상태를 정리해야 함

→ Table API만으로는 이런 "이벤트 간 패턴 매칭"을 표현하기 어렵습니다.

### Table API vs DataStream API 비교

| 특징 | Table API (flink_job.py) | DataStream API (flink_fraud.py) |
|------|--------------------------|--------------------------------|
| **추상화 수준** | 고수준 (SQL 스타일) | 저수준 (프로그래밍 스타일) |
| **코드 스타일** | 선언적 (무엇을) | 명령형 (어떻게) |
| **적합한 용도** | 윈도우 집계, JOIN, GROUP BY | 복잡한 이벤트 패턴 탐지, CEP |
| **상태 관리** | 자동 (SQL 뒤에 숨김) | 수동 (ValueState 직접 관리) |
| **타이머** | 지원 안 함 | ProcessFunction으로 직접 제어 |
| **실무 예시** | "10초마다 사용자별 결제 총액" | "소액 후 1분 이내 고액 결제 = 사기" |

> 💡 **핵심**: Table API = "무엇을", DataStream API = "어떻게"

---
## 2. 사기 탐지 시나리오 상세

### 실제 카드 사기 패턴

현실에서 카드 사기범들이 많이 사용하는 전형적인 패턴:

**1단계**: 소액 결제로 카드 유효성 확인 (100~900원)
- "이 카드가 정상적으로 작동하는가?"를 테스트
- 금액이 작아서 카드 소유자가 눈치채기 어려움

**2단계**: 유효성 확인 후 즉시 고액 결제 시도 (15000~50000원)
- 유효한 카드임을 확인했으므로 큰 금액 사용
- 시간차: 실제로는 수 분~수 시간이지만, 수업 시연용으로 5~30초로 단축

### stream-lab 환경 구성

![](https://cdn.discordapp.com/attachments/1457516082071081045/1470657461638926399/AOI_d_9TiRKBnMVUdX3Og6VT7RzLd-laQhtm8qKrmyePu46esz4uDu921LDRdfZYuF3txXx7Pvd-pkWUfrcQf-0hcLVyF5tkFOCDKM802L0HO7uWYRxKy4pMe533oH8-VHVzitGSrXoxiqakafgmUjqSqCSOkFNvtn1vAIIa0IdhU7Rc2aszs1024-rj.png?ex=698c17fe&is=698ac67e&hm=582b2d19ed38ef39e4d0572659bdb044912334b00d48f853fd4897e7e4e62dcc&)

### ASCII 시나리오 다이어그램

```
시간 ────────────────────────────────────────▶

[사기 패턴]
  10:00:05 소액 결제 (500원) ─┐
                             ├─ 20초 시간차
  10:00:25 고액 결제 (30000원)┘
           └─▶ 🚨 FRAUD ALERT!

[정상 패턴 1]
  10:01:10 소액 결제 (800원) ─┐
                             ├─ 70초 시간차 (1분 초과)
  10:02:20 고액 결제 (25000원)┘
           └─▶ ✅ 정상 (타이머 만료)

[정상 패턴 2]
  10:03:00 일반 결제 (3000원)
           └─▶ ✅ 정상 (소액도 고액도 아님)
```

---
## 3. 사기 패턴 생성기 (fraud_generator.py) 코드 워크스루

기존 `producer.py`(Day28_01)를 기반으로 사기 패턴을 주입하는 생성기입니다.

In [ ]:
# ===== fraud_generator.py 핵심 부분 =====

import heapq
import time
import random
from datetime import datetime

# 설정: 10%의 이벤트에 사기 패턴 주입
FRAUD_RATE = 0.1

# 왜: 사기 패턴의 "고액 결제"를 미래 시점에 예약하기 위해
# heapq(최소 힙)를 사용한다. Day28_01의 Chaos 패턴 재사용.
scheduled = []

# ===== 메인 루프 =====
while True:
    now = time.time()

    # 1. 예약 큐에서 전송 시각이 된 이벤트를 꺼내서 전송
    #    왜: 사기 패턴의 "고액 결제"는 소액 결제 후 5~30초 뒤에 예약되어 있다.
    #    heapq 덕분에 scheduled[0]이 항상 가장 빨리 보내야 할 이벤트이다.
    while scheduled:
        earliest_send_time, _earliest_event = scheduled[0]  # peek
        if earliest_send_time > now:
            break
        _send_time, event = heapq.heappop(scheduled)  # pop
        event["event_time"] = datetime.now().isoformat(timespec="milliseconds")
        # send_event("[FRAUD-LARGE]", event)

    # 2. 새 이벤트 생성
    user_id = f"U{random.randint(1, 5)}"
    event_time = datetime.now().isoformat(timespec="milliseconds")

    # 3. 사기 패턴 주입 여부 결정
    if random.random() < FRAUD_RATE:
        # ──────────────────────────────────────────────────────
        # 사기 패턴 (Fraud Pattern)
        # 왜: 실제 카드 사기의 전형적 패턴을 시뮬레이션한다.
        #   1단계: 소액 결제로 카드 유효성 확인 (100~900원)
        #   2단계: 유효성 확인 후 고액 결제 시도 (15000~50000원)
        #   시간차: 5~30초 (실제로는 수 분~수 시간이지만 수업 시연용으로 단축)
        #
        # FraudDetector(flink_fraud.py)가 이 패턴을 탐지해야 한다:
        #   "같은 user_id에서 소액(<1000) 후 1분 이내 고액(>10000) 발생"
        # ──────────────────────────────────────────────────────
        small_amount = random.randint(100, 900)
        small_event = {
            "user_id": user_id,
            "amount": small_amount,
            "event_time": event_time,
        }
        # send_event("[FRAUD-SMALL]", small_event)

        # 고액 결제를 5~30초 뒤로 예약
        delay = random.uniform(5, 30)
        large_amount = random.randint(15000, 50000)
        large_event = {
            "user_id": user_id,  # 왜: 반드시 같은 user_id여야 사기 패턴으로 탐지됨
            "amount": large_amount,
            "event_time": "",  # 실제 전송 시점에 갱신
        }
        send_time = time.time() + delay
        heapq.heappush(scheduled, (send_time, large_event))

        print(
            f"  └─ [FRAUD] {user_id}: 소액({small_amount}원) 전송 완료, "
            f"고액({large_amount}원) {delay:.1f}초 후 예약됨"
        )
    else:
        # ──────────────────────────────────────────────────────
        # 정상 결제 (Normal Payment)
        # 왜: 대부분의 결제는 정상이어야 사기 탐지의 의미가 있다.
        # 금액 범위(500~5000)는 사기 패턴의 임계값 사이에 위치.
        # ──────────────────────────────────────────────────────
        normal_amount = random.randint(500, 5000)
        normal_event = {
            "user_id": user_id,
            "amount": normal_amount,
            "event_time": event_time,
        }
        # send_event("[NORMAL]", normal_event)

    time.sleep(1)  # 1초마다 이벤트 생성

### heapq 패턴 복습 (Day28_01에서 배운 것)

**왜 heapq를 사용하는가?**
- 사기 패턴의 "고액 결제"를 미래 시점에 예약해야 함
- 예약된 이벤트 중 **가장 빨리 보내야 할 것**을 효율적으로 찾기 위해

**heapq의 특성**:
- `heapq.heappush(scheduled, (send_time, event))`: 전송 시각 순으로 자동 정렬
- `scheduled[0]`: 항상 가장 빨리 보내야 할 이벤트 (peek)
- `heapq.heappop(scheduled)`: 가장 빨리 보내야 할 이벤트를 꺼냄 (pop)

**시간 복잡도**:
- 단순 리스트 정렬: O(n log n)
- heapq: O(log n) (삽입/삭제 모두)

→ 고성능 실시간 처리에 필수적인 자료구조

---
## 4. Flink DataStream API 핵심 개념

### 4.1 KeyedStream: 사용자별 분리 처리

**KeyedStream이란?**
- `key_by(user_id)`로 같은 사용자의 이벤트를 **같은 파티션**으로 모음
- 각 파티션(키)마다 **독립적인 상태**를 유지할 수 있음
- Flink가 자동으로 키별 병렬 처리 수행

**왜 필요한가?**
- 사기 탐지는 **사용자별**로 패턴을 추적해야 함
- U1의 소액 결제 기록과 U2의 소액 결제 기록은 완전히 분리되어야 함

#### ASCII 다이어그램: KeyedStream

```
┌─────────────────────────────────────────────────────┐
│ DataStream (섞여 있는 이벤트들)                         │
│ U1:500원 → U2:300원 → U1:20000원 → U3:100원           │
└─────────────────────────────────────────────────────┘
                      ↓
                 key_by(user_id)
                      ↓
┌─────────────────────────────────────────────────────┐
│ KeyedStream (사용자별 분리)                            │
│                                                     │
│ [U1 파티션] U1:500원 → U1:20000원                      │
│                                                     │
│ [U2 파티션] U2:300원                                  │
│                                                     │
│ [U3 파티션] U3:100원                                  │
└─────────────────────────────────────────────────────┘
                      ↓
            각 파티션마다 독립 상태 유지
            (U1의 상태 ≠ U2의 상태)
```

### 4.2 ValueState: 사용자별 기억 저장소

**ValueState란?**
- 키(여기서는 user_id)마다 **하나의 값**을 저장하는 상태
- Flink가 자동으로 키별로 분리하여 관리
- 체크포인트에 포함되어 장애 시에도 유지됨

**일반 변수와의 차이**:

| 구분 | 일반 변수 (self.last_small) | ValueState |
|------|----------------------------|-----------|
| **스코프** | 인스턴스 전체 공유 | 키(user_id)별 독립 |
| **장애 복구** | 유실됨 | 체크포인트로 복구 |
| **병렬 처리** | 동시성 문제 발생 | Flink가 자동 관리 |
| **메모리 관리** | 수동 정리 필요 | TTL 등 자동 관리 가능 |

**예시**:
```python
# ❌ 잘못된 방식: 일반 변수
class FraudDetector(KeyedProcessFunction):
    def __init__(self):
        self.last_small = {}  # {user_id: amount} 딕셔너리

    def process_element(self, value, ctx):
        user_id = value.user_id
        # 문제: 모든 키가 같은 딕셔너리를 공유 → 장애 시 유실, 체크포인트 미포함
        self.last_small[user_id] = value.amount

# ✅ 올바른 방식: ValueState
class FraudDetector(KeyedProcessFunction):
    def open(self, runtime_context):
        # Flink 관리 상태: 키별 자동 분리, 체크포인트 포함
        self._small_amount_state = runtime_context.get_state(
            ValueStateDescriptor("small_amount", Types.INT())
        )

    def process_element(self, value, ctx):
        # 현재 키(user_id)의 상태만 접근
        last_small = self._small_amount_state.value()  # U1의 상태 ≠ U2의 상태
        self._small_amount_state.update(value.amount)
```

### 4.3 ProcessFunction + Timer: 시간 기반 콜백

**ProcessFunction이란?**
- DataStream API에서 가장 **저수준의 처리 함수**
- 이벤트마다 `process_element()` 호출
- 타이머를 등록하여 미래 시점에 `on_timer()` 콜백 받을 수 있음

**KeyedProcessFunction**:
- ProcessFunction + KeyedStream의 결합
- **키별 독립적인 상태**와 **키별 독립적인 타이머** 사용 가능

**Timer의 필요성**:
- "소액 결제 후 1분 이내에 고액 결제가 없으면 정상"
- → 1분이 지나면 상태를 정리해야 메모리 누수 방지
- → `register_processing_time_timer(now + 60초)`로 타이머 등록
- → 1분 후 `on_timer()` 자동 호출 → 상태 정리

#### 타이머 종류

| 타이머 종류 | 기준 시각 | 용도 |
|------------|---------|------|
| **Processing Time** | 서버의 현재 시각 | 실시간 알림, 세션 타임아웃 |
| **Event Time** | 이벤트의 event_time | 정확한 윈도우 종료 감지 |

> 💡 **사기 탐지는 Processing Time 사용**: "소액 결제 처리 시점"부터 1분을 세기 때문

### 4.4 Keyed State vs Operator State 비교

Flink의 상태는 크게 두 가지로 나뉩니다.

| 구분 | Keyed State | Operator State |
|------|-------------|----------------|
| **스코프** | 키(user_id)별 독립 | Operator 인스턴스 전체 공유 |
| **접근 방법** | KeyedProcessFunction | CheckpointedFunction |
| **병렬화** | 키별 자동 분산 | 병렬도만큼 복제 |
| **상태 타입** | ValueState, ListState, MapState | ListState, UnionListState, BroadcastState |
| **실무 예시** | 사용자별 세션 추적, 사기 탐지 | Kafka Offset, 전역 설정 |

**우리가 사용할 것: Keyed State**
- 사용자별로 독립적인 상태를 유지해야 하므로 ValueState 사용
- Flink가 자동으로 키별 분산 처리 수행

---
## 5. 사기 탐지 Job 구현 (flink_fraud.py) 전체 코드 워크스루

### 전체 파이프라인 구조

```
KafkaSource (fraud-payments)
      ↓
ParseEventFunction (JSON → Row)
      ↓
key_by(user_id)
      ↓
FraudDetector (KeyedProcessFunction)
  - 소액 결제 → 상태 저장 + 타이머 등록
  - 고액 결제 → 상태 확인 → Alert 발생
  - 타이머 만료 → 상태 정리
      ↓
print() + JdbcSink (PostgreSQL)
```

### 5.1 환경 설정 및 Source

In [ ]:
from pyflink.common import Types, WatermarkStrategy
from pyflink.datastream import StreamExecutionEnvironment, RuntimeExecutionMode
from pyflink.datastream.connectors.kafka import KafkaSource, KafkaOffsetsInitializer
from pyflink.common.serialization import SimpleStringSchema

# ===== 1. 실행 환경 설정 =====
env = StreamExecutionEnvironment.get_execution_environment()
# 왜: STREAMING 모드를 명시적으로 설정한다.
# DataStream API의 기본 모드이지만, 교육 목적으로 명시.
env.set_runtime_mode(RuntimeExecutionMode.STREAMING)
# 왜: 장애 복구용 체크포인트. flink_job.py(Table API)와 동일한 설정.
env.enable_checkpointing(10 * 1000)

# ===== 2. Kafka Source =====
# 왜: DataStream API에서는 KafkaSource 빌더로 소스를 직접 구성한다.
# Table API(flink_job.py)에서는 CREATE TABLE DDL로 선언했던 것과 대비된다.
#
# Table API:  CREATE TABLE payment_source (...) WITH ('connector' = 'kafka', ...)
# DataStream: KafkaSource.builder().set_topics(...).build()
kafka_source = (
    KafkaSource.builder()
    .set_bootstrap_servers("kafka:9092")
    .set_topics("fraud-payments")
    .set_group_id("flink-fraud-detector")
    .set_starting_offsets(KafkaOffsetsInitializer.latest())
    .set_value_only_deserializer(SimpleStringSchema())
    .build()
)

# 왜: WatermarkStrategy.no_watermarks()를 사용한다.
# 이 잡은 Processing Time 기반 타이머를 사용하므로 Event Time Watermark가 필요 없다.
# flink_job.py(Table API)에서는 WATERMARK FOR event_time으로 Event Time을 쓰지만,
# 여기서는 "소액 후 1분 이내 고액"이라는 Processing Time 기반 패턴을 탐지한다.
ds = env.from_source(
    kafka_source,
    WatermarkStrategy.no_watermarks(),
    "Kafka Fraud Payments Source",
)

**비교: Table API vs DataStream API의 Source 정의**

| 항목 | Table API (flink_job.py) | DataStream API (flink_fraud.py) |
|------|--------------------------|--------------------------------|
| **정의 방식** | SQL DDL (`CREATE TABLE`) | Python 빌더 패턴 (`KafkaSource.builder()`) |
| **Watermark** | DDL 안에 포함 (`WATERMARK FOR event_time`) | `WatermarkStrategy` 객체로 별도 지정 |
| **Deserialization** | DDL 안에 포함 (`'format' = 'json'`) | `SimpleStringSchema()` 명시 → 직접 파싱 |
| **코드량** | 짧음 (SQL 한 줄) | 길지만 세밀한 제어 가능 |

### 5.2 JSON 파싱

In [ ]:
import json
from pyflink.common import Row
from pyflink.datastream.functions import MapFunction

# ===== JSON 파싱 함수 =====
# 왜: Kafka에서 읽은 원시 문자열을 구조화된 Row로 변환해야
# KeyedProcessFunction에서 user_id, amount 등의 필드에 접근할 수 있다.
class ParseEventFunction(MapFunction):
    def map(self, value):
        data = json.loads(value)
        return Row(
            user_id=str(data.get("user_id", "")),
            amount=int(data.get("amount", 0)),
            event_time=str(data.get("event_time", "")),
        )

# ===== 3. JSON 파싱 =====
# 왜: Kafka에서 읽은 raw JSON 문자열을 Row(user_id, amount, event_time)으로 변환.
# Table API에서는 'format' = 'json'으로 자동 파싱되지만,
# DataStream API에서는 MapFunction으로 직접 파싱해야 한다.
parsed = ds.map(
    ParseEventFunction(),
    output_type=Types.ROW_NAMED(
        ["user_id", "amount", "event_time"],
        [Types.STRING(), Types.INT(), Types.STRING()],
    ),
)

**왜 직접 파싱하는가?**
- Table API: `'format' = 'json'` → Flink가 자동으로 JSON을 테이블 행으로 변환
- DataStream API: `SimpleStringSchema()` → 원시 문자열로 읽음 → MapFunction으로 직접 파싱

**장점**:
- 더 복잡한 JSON 구조 처리 가능 (중첩 JSON, 조건부 필드 등)
- 파싱 에러를 직접 핸들링 가능 (로깅, Side Output 등)

### 5.3 핵심: FraudDetector (KeyedProcessFunction)

사기 탐지 로직의 **핵심**입니다. 상태 전이를 먼저 이해하고 코드를 봅시다.

#### 상태 전이 다이어그램

![](https://cdn.discordapp.com/attachments/1457516082071081045/1470657575963070689/AOI_d_-nvveVWSbMoYvvZZy591H-1yyLGoBYu5fmfctxqq_r_h2PjH6LhEJHHJEHv7YT7xpQiJ_RD1x4S0GDcNoZLRHEmuKwPgZQ7lx3Ewu4q0aEF9J19QHIZupDkmjjgxcscKR0PznwAK8Q94iEStEL5m6Lnt_1yKhzbijZtfuPVnbPDP_gjAs1024-rj.png?ex=698c1819&is=698ac699&hm=4d03e64396ced6f40eda86a84600897a0d6a33db603ee5ccb3aa740d7c8e109c&)

#### FraudDetector 전체 코드

In [ ]:
from pyflink.datastream.functions import KeyedProcessFunction, RuntimeContext
from pyflink.datastream.state import ValueStateDescriptor

# 사기 패턴 임계값 (fraud_generator.py와 일치)
SMALL_AMOUNT_THRESHOLD = 1000   # 이 미만이면 "소액 결제"
LARGE_AMOUNT_THRESHOLD = 10000  # 이 이상이면 "고액 결제"
TIMER_DURATION_MS = 60 * 1000   # 1분 (소액→고액 패턴의 시간 윈도우)


# ===== 핵심: 사기 탐지 KeyedProcessFunction =====
#
KeyedProcessFunction이란?
#   - key_by()로 분류된 각 키(여기서는 user_id)마다 독립적으로 동작
#   - 키별 상태(ValueState)를 유지 → U1의 상태와 U2의 상태는 완전히 분리
#   - Processing Time Timer를 등록해 미래 시점에 콜백을 받을 수 있음
#
# 탐지 로직:
#   1. 소액 결제(<1000) 도착 → 상태에 금액 저장 + 1분 타이머 등록
#   2. 고액 결제(>10000) 도착 + 상태에 소액 기록 있음 → FRAUD ALERT!
#   3. 타이머 만료(1분 경과) → 소액 기록 삭제 (패턴 실패, 정상으로 간주)
#
# 왜 이 방식이 효과적인가:
#   - Flink가 상태를 Checkpoint로 관리하므로 장애 시에도 상태가 유실되지 않음
#   - key_by(user_id) 덕분에 수백만 사용자를 병렬 처리 가능
#   - 타이머가 자동으로 패턴 만료를 관리 → 메모리 누수 방지
# ================================================================
class FraudDetector(KeyedProcessFunction):

    def __init__(self):
        # 왜: __init__에서는 선언만 한다.
        # 실제 State 초기화는 open()에서 해야 Flink 런타임이 관리하는 State를 받을 수 있다.
        self._small_amount_state = None  # 소액 결제 금액 기억용
        self._timer_state = None         # 등록한 타이머 시각 기억용

    def open(self, runtime_context: RuntimeContext):
        # 왜: ValueStateDescriptor로 Flink 관리 상태를 생성한다.
        # 이 상태는 Checkpoint에 포함되어 장애 복구 시에도 유지된다.
        #
        # Table API(flink_job.py)에서는 이런 상태 관리가 SQL 뒤에 숨겨져 있지만,
        # DataStream API에서는 개발자가 직접 선언하고 관리한다.
        self._small_amount_state = runtime_context.get_state(
            ValueStateDescriptor("small_amount", Types.INT())
        )
        self._timer_state = runtime_context.get_state(
            ValueStateDescriptor("timer_ts", Types.LONG())
        )

    def process_element(self, value, ctx: KeyedProcessFunction.Context):
        """
        이벤트가 도착할 때마다 호출된다.
        value: Row(user_id, amount, event_time)
        ctx: 타이머 서비스, 현재 키 등에 접근할 수 있는 컨텍스트
        """
        amount = value.amount
        user_id = value.user_id

        # ── 케이스 1: 고액 결제 도착 ──
        if amount >= LARGE_AMOUNT_THRESHOLD:
            small_amount = self._small_amount_state.value()
            if small_amount is not None:
                # ===== FRAUD DETECTED! =====
                # 왜: 상태에 소액 기록이 있다 = 이전에 소액 결제가 있었고, 아직 1분이 안 지남
                # → "소액 후 고액" 패턴 성립 → 사기 알림 발생
                print(
                    f"[ALERT] 사기 탐지! user={user_id} "
                    f"소액={small_amount}원 → 고액={amount}원"
                )
                yield Row(user_id=user_id, small_amount=small_amount, large_amount=amount)

            # 왜: 사기든 아니든, 고액 결제 후에는 상태를 정리한다.
            # 같은 사용자의 다음 패턴을 새로 추적하기 위함.
            self._clean_up(ctx.timer_service())

        # ── 케이스 2: 소액 결제 도착 ──
        if amount < SMALL_AMOUNT_THRESHOLD:
            # 왜: 소액 결제를 상태에 기록하고, 1분 타이머를 등록한다.
            # 이 타이머가 만료되기 전에 고액 결제가 오면 → 사기
            # 타이머가 먼저 만료되면 → 정상 (on_timer에서 상태 정리)
            self._small_amount_state.update(amount)
            timer_ts = ctx.timer_service().current_processing_time() + TIMER_DURATION_MS
            ctx.timer_service().register_processing_time_timer(timer_ts)
            self._timer_state.update(timer_ts)

    def on_timer(self, timestamp: int, ctx: KeyedProcessFunction.OnTimerContext):
        """
        등록한 타이머가 만료되면 호출된다.
        왜: 소액 결제 후 1분이 지나도 고액 결제가 없으면 정상 거래로 간주.
        상태를 정리하여 메모리를 확보한다.
        """
        self._small_amount_state.clear()
        self._timer_state.clear()

    def _clean_up(self, timer_service):
        """상태와 타이머를 모두 정리하는 헬퍼 메서드"""
        timer_ts = self._timer_state.value()
        if timer_ts is not None:
            timer_service.delete_processing_time_timer(timer_ts)
        self._small_amount_state.clear()
        self._timer_state.clear()

#### 코드 상세 설명

**1. `open()` 메서드**
```python
def open(self, runtime_context: RuntimeContext):
    self._small_amount_state = runtime_context.get_state(
        ValueStateDescriptor("small_amount", Types.INT())
    )
```
- `open()`은 Task가 시작될 때 한 번만 호출됨
- `ValueStateDescriptor`로 상태의 이름과 타입을 정의
- `runtime_context.get_state()`로 Flink 관리 상태를 받음
- **왜 `__init__`이 아닌 `open()`에서?** Flink 런타임이 `runtime_context`를 주입하기 전에는 상태를 생성할 수 없음

**2. `process_element()` 메서드**
- 이벤트 하나마다 호출되는 핵심 로직
- `value`: 현재 처리 중인 이벤트 (Row 타입)
- `ctx`: 타이머 서비스, 현재 키, 현재 시각 등에 접근
- `yield Row(...)`: 다음 스트림으로 결과 전달 (Python Generator 패턴)

**3. `on_timer()` 메서드**
- `register_processing_time_timer()`로 등록한 타이머가 만료되면 호출
- `timestamp`: 타이머 등록 시 지정한 시각
- 여기서는 "1분 경과 → 상태 정리" 용도

**4. 타이머 관리의 중요성**
- 타이머를 등록만 하고 삭제 안 하면? → 메모리 누수
- `_timer_state`에 타이머 시각을 저장하는 이유? → 고액 결제 도착 시 타이머를 삭제하기 위해
- `delete_processing_time_timer(timer_ts)`로 명시적 삭제 필요

### 5.4 KeyBy + Process

In [ ]:
# ===== 4. KeyBy + FraudDetector =====
# 왜: key_by(user_id)로 같은 사용자의 이벤트를 같은 파티션으로 모은다.
# 이렇게 해야 FraudDetector가 사용자별 독립 상태를 유지할 수 있다.
#
# Table API에서 GROUP BY user_id와 비슷하지만,
# 여기서는 집계가 아니라 "이벤트 간 패턴 매칭"을 수행한다.
alerts = parsed.key_by(lambda row: row.user_id).process(
    FraudDetector(),
    output_type=Types.ROW_NAMED(
        ["user_id", "small_amount", "large_amount"],
        [Types.STRING(), Types.INT(), Types.INT()],
    ),
)

**key_by()의 동작 원리**:
- `lambda row: row.user_id`: 각 이벤트에서 키를 추출하는 함수
- Flink가 이 키로 해시 파티셔닝 수행 → 같은 user_id는 같은 TaskManager로 전송
- 각 파티션마다 FraudDetector 인스턴스가 독립적으로 동작

**병렬 처리 예시**:
```
TaskManager 1:  U1, U3 처리 (FraudDetector 인스턴스 A)
TaskManager 2:  U2, U4 처리 (FraudDetector 인스턴스 B)
TaskManager 3:  U5 처리     (FraudDetector 인스턴스 C)
```
→ 인스턴스 A의 상태 ≠ 인스턴스 B의 상태

### 5.5 Print Sink + JDBC Sink

In [ ]:
from pyflink.datastream.connectors.jdbc import (
    JdbcSink,
    JdbcConnectionOptions,
    JdbcExecutionOptions,
)

# ===== 5. Print Sink (콘솔 출력) =====
# 왜: 수업 시연 시 docker compose logs로 실시간 알림을 확인할 수 있다.
alerts.print()

# ===== 6. JDBC Sink (PostgreSQL 저장) =====
# 왜: 탐지된 사기 알림을 fraud_alerts 테이블에 저장한다.
# Table API에서는 CREATE TABLE sink_ddl WITH ('connector'='jdbc')로 선언했지만,
# DataStream API에서는 JdbcSink.sink()로 직접 구성한다.
#
# Table API:  INSERT INTO flink_results SELECT ...
# DataStream: alerts.add_sink(JdbcSink.sink("INSERT INTO ...", ...))
JDBC_URL = "jdbc:postgresql://postgres:5432/streamdb"

jdbc_sink = JdbcSink.sink(
    "INSERT INTO fraud_alerts (user_id, small_amount, large_amount) VALUES (?, ?, ?)",
    type_info=Types.ROW_NAMED(
        ["user_id", "small_amount", "large_amount"],
        [Types.STRING(), Types.INT(), Types.INT()],
    ),
    jdbc_connection_options=(
        JdbcConnectionOptions.JdbcConnectionOptionsBuilder()
        .with_url(JDBC_URL)
        .with_driver_name("org.postgresql.Driver")
        .with_user_name("postgres")
        .with_password("postgres")
        .build()
    ),
    jdbc_execution_options=(
        JdbcExecutionOptions.builder()
        .with_batch_interval_ms(1000)
        .with_batch_size(10)
        .with_max_retries(3)
        .build()
    ),
)
alerts.add_sink(jdbc_sink)

# ===== 7. 실행 =====
# 왜: DataStream API에서는 env.execute()로 잡을 제출한다.
# Table API에서는 t_env.execute_sql(insert_sql).wait()로 실행했던 것과 대비.
# env.execute("Fraud Detection Job")

**JdbcSink 설정 설명**:

| 설정 | 값 | 의미 |
|------|---|------|
| **batch_interval_ms** | 1000 | 1초마다 배치로 DB에 쓰기 (성능 최적화) |
| **batch_size** | 10 | 10개 이벤트마다 배치로 DB에 쓰기 |
| **max_retries** | 3 | 실패 시 최대 3회 재시도 |

**왜 배치로 쓰는가?**
- 사기 알림 하나마다 DB에 쓰면 → 너무 많은 DB 연결 → 성능 저하
- 배치로 모아서 쓰면 → 적은 DB 연결 → 높은 처리량
- 단, 1초 지연 발생 → 실시간성과 처리량의 트레이드오프

---
## 6. 실행 및 결과 확인

### 실행 명령어

stream-lab에서 제공하는 `just` 명령어로 간편하게 실행할 수 있습니다.

```bash
# 1. 전체 환경 초기화 + Fraud Detection 실행
just phase-fraud
```

**내부 동작 과정** (`justfile` 참고):

```
1. 전체 컨테이너 정리 (just clean)
2. Kafka + PostgreSQL 시작 (just infra)
3. Flink 클러스터 시작 (jobmanager + taskmanager)
4. Kafka 토픽(fraud-payments) 생성
5. Flink Job 제출 (flink_fraud.py)
6. Fraud Generator 시작 (fraud_generator.py)
```

### Docker Compose 직접 실행 (참고용)

```bash
# 1. 인프라 시작
docker compose up -d kafka postgres

# 2. Flink 클러스터 시작
docker compose up -d jobmanager taskmanager

# 3. Kafka 토픽 생성
docker compose exec kafka /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create --topic fraud-payments \
  --partitions 1 --replication-factor 1 \
  --if-not-exists

# 4. Flink Job 제출
docker compose exec jobmanager flink run -d -py /opt/flink/src/flink_fraud.py

# 5. Fraud Generator 시작
docker compose up -d fraud-generator
```

### 결과 확인

#### 1. 실시간 로그 확인

```bash
# Flink TaskManager 로그에서 [ALERT] 메시지 확인
docker compose logs -f taskmanager | grep ALERT
```

**예상 출력**:
```
[ALERT] 사기 탐지! user=U2 소액=850원 → 고액=32000원
[ALERT] 사기 탐지! user=U1 소액=300원 → 고액=45000원
[ALERT] 사기 탐지! user=U4 소액=650원 → 고액=28000원
```

#### 2. DB 조회

```bash
# fraud_alerts 테이블 조회
just query-fraud
```

**예상 결과**:

| alert_id | user_id | small_amount | large_amount | alert_time |
|----------|---------|--------------|--------------|------------|
| 1 | U2 | 850 | 32000 | 2026-02-10 10:05:23 |
| 2 | U1 | 300 | 45000 | 2026-02-10 10:06:15 |
| 3 | U4 | 650 | 28000 | 2026-02-10 10:07:42 |
| 4 | U3 | 900 | 19000 | 2026-02-10 10:08:30 |

**사용자별 사기 탐지 횟수**:

| user_id | alert_count | avg_large_amount |
|---------|-------------|------------------|
| U2 | 3 | 35000 |
| U1 | 2 | 40000 |
| U4 | 2 | 25000 |
| U3 | 1 | 19000 |

> 💡 **해석**: FRAUD_RATE=0.1이므로 전체 이벤트의 약 10%가 사기 패턴입니다.

### Flink Web UI 확인

- URL: http://localhost:8081
- **Running Jobs** → "Fraud Detection Job" 클릭
- **Metrics** 탭에서 처리량, 백프레셔 등 확인

**확인할 지표**:
- `numRecordsIn`: Kafka에서 읽은 총 이벤트 수
- `numRecordsOut`: FraudDetector가 출력한 알림 수 (≈ 10%)
- `State Size`: ValueState가 차지하는 메모리 (타이머 만료로 정리되는지 확인)

---
## 7. Table API vs DataStream API 최종 비교

### 비교표

| 비교 항목 | Table API (flink_job.py) | DataStream API (flink_fraud.py) |
|----------|--------------------------|--------------------------------|
| **추상화 수준** | 고수준 (SQL 스타일) | 저수준 (프로그래밍 스타일) |
| **코드 스타일** | 선언적 (WHAT) - "10초 윈도우로 집계해줘" | 명령형 (HOW) - "이벤트마다 이렇게 처리해" |
| **적합한 용도** | 윈도우 집계, JOIN, GROUP BY, 간단한 필터링 | 복잡한 이벤트 패턴 탐지, CEP, 커스텀 로직 |
| **상태 관리** | 자동 (SQL 뒤에 숨김) | 수동 (ValueState 직접 관리) |
| **타이머** | 지원 안 함 | ProcessFunction으로 직접 제어 |
| **코드량** | 짧음 (SQL 몇 줄) | 김 (클래스 + 메서드) |
| **학습 곡선** | 낮음 (SQL 알면 됨) | 높음 (Flink 내부 개념 이해 필요) |
| **실무 예시** | "10초마다 사용자별 결제 총액" | "소액 후 1분 이내 고액 = 사기" |
| **성능 최적화** | Flink가 자동 최적화 | 개발자가 직접 튜닝 |
| **디버깅** | 어려움 (SQL 실행 계획 분석) | 쉬움 (print 디버깅, 로그) |

### 언제 무엇을 쓸까?

#### Table API를 선택하는 경우

✅ **윈도우 집계**: "최근 5분간 평균 주문 금액"
✅ **JOIN**: "주문 테이블과 상품 테이블을 결합"
✅ **GROUP BY + 집계 함수**: "지역별, 시간대별 매출 합계"
✅ **빠른 프로토타이핑**: SQL로 빠르게 검증 후 DataStream으로 고도화
✅ **팀에 SQL 전문가가 많을 때**

#### DataStream API를 선택하는 경우

✅ **이벤트 간 패턴 매칭**: "A 이벤트 후 B 이벤트가 오면 알림"
✅ **상태 + 타이머 조합**: "세션 타임아웃", "사기 패턴 만료"
✅ **복잡한 비즈니스 로직**: SQL로 표현 불가능한 조건문, 반복문
✅ **CEP (Complex Event Processing)**: 순서 있는 이벤트 시퀀스 탐지
✅ **커스텀 Source/Sink**: 특수한 외부 시스템 연동

> 💡 **실무 팁**: Table API로 시작 → 한계 도달 → DataStream API로 전환

### 코드 비교: 같은 작업, 다른 접근

**예시**: "사용자별 최근 결제 금액 추적"

#### Table API 방식 (간결하지만 제한적)

```sql
-- 최근 10초 윈도우로 사용자별 총액 (간단!)
SELECT
  window_start,
  user_id,
  SUM(amount) AS total_amount
FROM TABLE(
  TUMBLE(TABLE payment_source, DESCRIPTOR(event_time), INTERVAL '10' SECOND)
)
GROUP BY window_start, window_end, user_id
```

#### DataStream API 방식 (복잡하지만 유연)

```python
# 사용자별 마지막 3개 결제 추적 (Table API로는 불가능!)
class Last3Tracker(KeyedProcessFunction):
    def open(self, runtime_context):
        # ListState: 여러 값을 리스트로 저장
        self._last3 = runtime_context.get_list_state(
            ListStateDescriptor("last3", Types.INT())
        )

    def process_element(self, value, ctx):
        amounts = list(self._last3.get())
        amounts.append(value.amount)
        if len(amounts) > 3:
            amounts.pop(0)  # 가장 오래된 것 제거
        self._last3.update(amounts)

        avg = sum(amounts) / len(amounts)
        if value.amount > avg * 3:
            yield Row(user_id=value.user_id, alert="급증")
```

---
## 📝 FAQ

**Q1. DataStream API가 Table API보다 성능이 더 좋나요?**

→ 꼭 그렇지는 않습니다. Table API는 Flink의 쿼리 최적화 엔진(Calcite)이 자동으로 실행 계획을 최적화합니다. DataStream API는 개발자가 직접 작성한 코드 그대로 실행되므로, 잘못 짜면 오히려 느릴 수 있습니다. **복잡한 로직이 필요할 때만** DataStream API를 선택하세요.

**Q2. 실무에서 사기 탐지는 더 복잡하지 않나요?**

→ 맞습니다. 실제 사기 탐지 시스템은 다음과 같은 고도화가 필요합니다:
- **머신러닝 모델 통합**: 단순 규칙 기반이 아닌 ML 점수 기반 탐지
- **다중 패턴 조합**: "소액 → 고액" 외에도 "동일 카드 여러 지역 동시 사용" 등
- **False Positive 감소**: 정상 거래를 사기로 오판하지 않도록 정교한 필터링
- **실시간 피드백 루프**: 탐지된 사기를 DB에 저장 → 재학습 → 모델 갱신

**Q3. CEP 라이브러리가 따로 있다고 들었는데요?**

→ 네, Flink는 **CEP (Complex Event Processing)** 라이브러리를 제공합니다.
```python
from pyflink.cep import Pattern, CEP

# "A 이벤트 → B 이벤트 → C 이벤트" 순서 패턴 탐지
pattern = Pattern.begin("start").where(lambda evt: evt.type == "A") \
                .next("middle").where(lambda evt: evt.type == "B") \
                .followedBy("end").where(lambda evt: evt.type == "C")

CEP.pattern(stream, pattern).select(lambda match: match)
```
CEP는 **순서가 있는 이벤트 시퀀스**를 탐지할 때 ProcessFunction보다 편리합니다. 하지만 PyFlink의 CEP는 Java/Scala 버전보다 기능이 제한적이므로, 복잡한 패턴은 DataStream API로 직접 구현하는 것이 일반적입니다.

**Q4. Table API와 DataStream API를 섞어 쓸 수 있나요?**

→ 가능합니다!
```python
# Table → DataStream 변환
table = t_env.from_path("payment_source")
ds = t_env.to_data_stream(table)
processed = ds.key_by(...).process(FraudDetector())

# DataStream → Table 변환
result_table = t_env.from_data_stream(processed)
t_env.execute_sql("INSERT INTO sink SELECT * FROM result_table")
```
실무에서는 **Table API로 전처리(필터링, 조인) → DataStream API로 복잡한 로직 → 다시 Table API로 Sink** 패턴을 많이 사용합니다.

**Q5. 타이머가 만료되지 않고 계속 쌓이면 메모리 누수 아닌가요?**

→ 좋은 질문입니다! 타이머는 다음과 같이 관리됩니다:
- **자동 정리**: `on_timer()` 호출 후 타이머는 자동으로 삭제됨
- **명시적 삭제**: `delete_processing_time_timer(timer_ts)`로 수동 삭제 가능 (FraudDetector에서 사용)
- **TTL (Time-To-Live)**: ValueState에 TTL을 설정하면 일정 시간 후 자동 삭제
  ```python
  state_descriptor = ValueStateDescriptor("small_amount", Types.INT())
  state_descriptor.enable_time_to_live(StateTtlConfig.new_builder(Time.minutes(5)).build())
  ```
- **체크포인트**: 타이머도 체크포인트에 포함되어 장애 복구 시 복원됨

---
## ✅ 퀴즈

**Q1.** DataStream API의 KeyedStream에서 `key_by(user_id)` 호출의 효과로 **올바르지 않은** 것은?

- A) 같은 user_id를 가진 이벤트가 같은 파티션으로 전송된다
- B) 각 파티션마다 독립적인 ValueState를 유지할 수 있다
- C) 모든 이벤트가 하나의 TaskManager로 집중되어 병렬 처리가 불가능하다
- D) Flink가 자동으로 해시 파티셔닝을 수행한다

<details>
<summary>정답 확인</summary>

**정답: C)**

`key_by()`는 키별로 이벤트를 **분산**시키므로 병렬 처리가 가능합니다. 예를 들어 5개의 TaskManager가 있다면, U1은 TM1, U2는 TM2, ... 이런 식으로 분산됩니다. 모든 이벤트가 한 곳으로 집중되는 것은 **잘못된 설명**입니다.

</details>

---

**Q2.** FraudDetector의 `process_element()`에서 소액 결제 도착 시 타이머를 등록하는 이유는?

- A) 고액 결제가 올 때마다 알림을 보내기 위해
- B) 1분이 지나도 고액 결제가 없으면 상태를 정리하기 위해
- C) 이벤트 처리 속도를 측정하기 위해
- D) Kafka에서 다음 이벤트를 가져오기 위해

<details>
<summary>정답 확인</summary>

**정답: B)**

타이머는 "소액 결제 후 1분이 지나면 정상 거래로 간주하고 상태를 정리"하기 위해 사용됩니다. 타이머가 만료되면 `on_timer()` 메서드가 호출되어 `_small_amount_state.clear()`로 상태를 정리합니다. 이렇게 하지 않으면 모든 소액 결제 기록이 메모리에 계속 쌓여 메모리 누수가 발생합니다.

</details>

---

**Q3.** Table API와 DataStream API의 비교로 **틀린** 것은?

- A) Table API는 SQL 스타일로 선언적이고, DataStream API는 명령형 프로그래밍 스타일이다
- B) Table API는 상태 관리를 자동으로 하지만, DataStream API는 ValueState를 직접 관리해야 한다
- C) Table API는 타이머를 지원하지 않지만, DataStream API는 ProcessFunction으로 타이머를 사용할 수 있다
- D) DataStream API는 항상 Table API보다 성능이 우수하므로 실무에서는 DataStream만 사용해야 한다

<details>
<summary>정답 확인</summary>

**정답: D)**

DataStream API가 항상 더 빠른 것은 아닙니다. Table API는 Flink의 쿼리 최적화 엔진(Calcite)이 자동으로 실행 계획을 최적화하므로, 간단한 집계/조인 작업에서는 오히려 Table API가 더 효율적일 수 있습니다. **복잡한 이벤트 패턴 탐지나 커스텀 로직이 필요할 때만** DataStream API를 선택해야 합니다.

</details>

---
## 📌 핵심 요약

| 개념 | 핵심 내용 |
|------|----------|
| **Table API vs DataStream** | Table = SQL 스타일 (무엇을), DataStream = 프로그래밍 스타일 (어떻게) |
| **KeyedStream** | `key_by(user_id)`로 사용자별 분리 처리, 키별 독립 상태 유지 |
| **ValueState** | 키별 하나의 값 저장, 체크포인트로 장애 복구 보장 |
| **ProcessFunction** | 이벤트마다 `process_element()` 호출, 타이머로 미래 시점 콜백 |
| **Timer** | `register_processing_time_timer()`로 등록, `on_timer()`로 만료 처리, 상태 정리에 필수 |
| **사기 탐지 패턴** | 소액(<1000) → 1분 이내 → 고액(>10000) = 사기 알림 |
| **실무 적용** | Table API로 시작 → 한계 도달 → DataStream으로 전환이 일반적 |


% [markdown]
---
---
## 다음 파일

# ⚡ 4교시: Flink 운영 — Checkpoint, Savepoint, 장애 복구, 스케일링

## 🎯 학습 목표
1. Checkpoint와 Savepoint의 차이를 이해하고 설명할 수 있다
2. Savepoint를 생성하고 복원하여 Job을 안전하게 재시작할 수 있다
3. Parallelism을 변경하여 Job을 스케일링할 수 있다
4. TaskManager 장애 시 자동 복구 과정을 확인할 수 있다

### 전제 조건
- Day28 1~3교시 완료
- stream-lab 환경 실행 가능 (`just phase4`로 Phase 4 기동)

---
## 1. 도입 — 운영 환경에서의 Flink

### 1.1 "코드를 배포했다"와 "운영한다"의 차이

개발 단계에서는 코드가 잘 동작하면 끝이지만, **운영 단계**에서는 전혀 다른 질문들이 등장합니다.

| 상황 | 질문 | 필요한 능력 |
|------|------|------------|
| 서버가 갑자기 죽으면? | 데이터를 유실하지 않고 복구할 수 있나? | **장애 복구** |
| 코드를 업데이트해야 한다면? | 서비스를 중단하지 않고 배포할 수 있나? | **무중단 배포** |
| 트래픽이 갑자기 증가하면? | 처리 용량을 늘릴 수 있나? | **스케일 조정** |
| 위 모든 상황에서 | 데이터 유실 없이 해결할 수 있나? | **상태 관리** |

운영 단계에서는 코드 자체보다 **상태 유지, 장애 복구, 변경 절차**가 중요합니다.
이번 실습은 Flink에서 이 세 가지를 실제 명령으로 검증하는 데 초점을 둡니다.


### === 운영에서 필요한 기능 ===

# ![](https://cdn.discordapp.com/attachments/1457516082071081045/1470658222393393295/image.png?ex=698f6473&is=698e12f3&hm=6fea08ed672097be4be4459f9d64ee40934addfba441aa21fb6596b7fdbbead4&)

---
## 2. Checkpoint vs Savepoint

Flink의 상태 관리에서 가장 중요한 두 가지 개념입니다.
둘 다 **"현재 상태를 저장한다"** 는 공통점이 있지만, **목적과 사용 방식이 완전히 다릅니다.**

### 2.1 Checkpoint — 자동 저장

Flink가 **자동으로** 주기적으로 생성하는 상태 스냅샷입니다.

- 목적: **장애 복구** (Flink 내부 용도)
- 특징: 주기적으로 자동 생성되어 실패 시 최근 일관된 상태로 복구

| 항목 | 설명 |
|------|------|
| 생성 주기 | 설정값 (예: 10초) |
| 생성 주체 | Flink (자동) |
| 저장 위치 | `state.checkpoints.dir` |
| 용도 | 장애 복구 |
| Job 종료 시 | 기본 삭제 (`RETAIN_ON_CANCELLATION` 설정 시 보존) |

**stream-lab의 `flink_job.py`에서 이미 설정된 내용:**

```python
# 10초마다 자동 저장
env.enable_checkpointing(CHECKPOINT_INTERVAL_SECONDS * 1000)  # 10_000ms

# RocksDB 상태 백엔드 (디스크 기반, 대용량 가능)
conf.set_string("state.backend.type", "rocksdb")
conf.set_string("state.checkpoints.dir", "file:///tmp/flink-checkpoints")

# Exactly-Once 보장
conf.set_string("execution.checkpointing.mode", "EXACTLY_ONCE")

# Job 취소 시에도 Checkpoint 보존
conf.set_string(
    "execution.checkpointing.externalized-checkpoint-retention",
    "RETAIN_ON_CANCELLATION",
)
```

### 2.2 Savepoint — 수동 저장

사용자가 **수동으로** 원하는 시점에 생성하는 상태 스냅샷입니다.

- 목적: **운영 작업** (버전 업그레이드, 스케일 변경, 마이그레이션)
- 특징: 운영자가 원하는 시점에 명시적으로 생성하고 장기 보관 가능

| 항목 | 설명 |
|------|------|
| 생성 시점 | 사용자 명령 시 |
| 생성 주체 | 사용자 (수동) |
| 저장 위치 | 지정된 경로 |
| 용도 | 운영 작업 (업그레이드, 스케일링, 마이그레이션) |
| Job 종료 시 | **항상 보존** (사용자가 직접 삭제) |

### 2.3 비교 정리표

| 구분 | Checkpoint | Savepoint |
|------|-----------|-----------|
| **생성 방식** | 시스템 자동 생성 | 운영자 수동 생성 |
| **생성** | 자동 (주기적) | 수동 (명령어) |
| **목적** | 장애 자동 복구 | 계획된 운영 작업 |
| **보관** | 일시적 (N개 유지) | 영구적 (직접 삭제) |
| **트리거** | `env.enable_checkpointing()` | `flink savepoint <job-id>` |
| **사용 시나리오** | TaskManager 크래시 | 코드 업데이트, 스케일 변경 |


# === Checkpoint vs Savepoint 타임라인 ===

# ![](https://cdn.discordapp.com/attachments/1457516082071081045/1470658873680593030/AOI_d_-UyHkZ2ZqlINTEIehMqJkXyfuT6yKujt9G3VJk0lMf5wTeL0e_jMBTh6pxJLMNk9T8e5k1oHvTfgr7U9oybJQGZZ_495pgXfTA6rNjDedhtXI2B0N9WNTuTgbqhHlSJAdRoAehkhCct1wN-9UpR0qSRcMqs6mJk6xFhhIZNU9I5j0Dwgs1024-rj.png?ex=698f650f&is=698e138f&hm=ea78c3818370b6e125e18afdd27652b7361526b12053cbd485e44783b379ea68&)

---
### 2.4 Flink Web UI에서 Checkpoint 확인하기

Flink Web UI(`http://localhost:8081`)에서 Checkpoint 상태를 실시간으로 확인할 수 있습니다.

**확인 경로:**
1. http://localhost:8081 접속
2. **Running Jobs** 클릭
3. 실행 중인 Job 선택
4. **Checkpoints** 탭 클릭

**확인할 수 있는 정보:**

| 항목 | 설명 |
|------|------|
| **Counts** | 트리거/진행중/완료/실패 횟수 |
| **Latest Completed Checkpoint** | 가장 최근 성공한 Checkpoint ID |
| **State Size** | 상태 데이터 크기 (RocksDB에 저장된 양) |
| **Duration** | Checkpoint 생성 소요 시간 |
| **Alignment Duration** | Barrier 정렬 시간 (Exactly-Once의 핵심) |

> 💡 **Alignment Duration이 길어지면?**
> → 데이터 편향(Skew)이 심하다는 의미. 일부 Task가 다른 Task보다 훨씬 많은 데이터를 처리하고 있을 수 있습니다.

---
## 3. Savepoint 실습

실제로 Savepoint를 생성하고, Job을 중지한 뒤, 복원해보겠습니다.

### 📌 전제: stream-lab의 flink_job이 실행 중이어야 합니다

```bash
# Phase 4가 실행 중이 아니라면
cd stream-lab
just phase4
# Flink Job이 RUNNING 상태가 될 때까지 약 2분 대기
```

### 3.1 현재 실행 중인 Job 확인

```bash
docker compose exec jobmanager flink list
```

출력 예시:
```
Printing result of job listing.
------------------- Running/Restarting Jobs -------------------
05.02.2026 12:34:56 : abc123def456789abc123def456789ab : insert-into_default... (RUNNING)
---------------------------------------------------------------
```

→ **Job ID** (`abc123def456789abc123def456789ab`)를 복사해두세요!

> 💡 **팁**: Job ID는 32자리 16진수 문자열입니다.

In [ ]:

# === 실습 명령어 정리 ===

# Step 1: Job 목록 확인
docker compose exec jobmanager flink list

# 출력에서 Job ID 확인:
# abc123def456789abc123def456789ab  ← 이 값을 <JOB_ID>로 사용

### 3.2 현재 결과 확인 (Savepoint 전 상태)

Savepoint 생성 전에 현재까지의 처리 결과를 확인합니다.

```bash
docker compose exec postgres psql -U postgres -d streamdb \
  -c "SELECT COUNT(*) AS row_count, SUM(total_amount) AS grand_total FROM flink_results;"
```

출력 예시:
```
 row_count | grand_total
-----------+-------------
       42  |       12345
```

→ 이 숫자를 기억하세요. 복원 후 이어서 처리되는지 확인할 것입니다.

### 3.3 Savepoint 생성

Job을 **중지하지 않고** Savepoint만 생성합니다.

```bash
# Savepoint 생성 (Job은 계속 실행됨)
docker compose exec jobmanager flink savepoint <JOB_ID> /tmp/flink-savepoints
```

출력 예시:
```
Savepoint completed. Path: file:///tmp/flink-savepoints/savepoint-abc123-1234567890
```

→ **Savepoint 경로**를 복사해두세요!

### 3.4 Job 중지 (Savepoint와 함께)

실무에서는 Savepoint 생성과 Job 중지를 **한 번에** 수행하는 것이 일반적입니다.

```bash
# 방법 1: stop (권장) — Savepoint 생성 + Job 중지를 한 번에
docker compose exec jobmanager flink stop --savepointPath /tmp/flink-savepoints <JOB_ID>
```

```bash
# 방법 2: cancel — Job 취소 (RETAIN_ON_CANCELLATION으로 마지막 Checkpoint 보존)
docker compose exec jobmanager flink cancel <JOB_ID>
```

| 방법 | Savepoint | Checkpoint | 권장 상황 |
|------|-----------|-----------|-----------|
| `flink stop` | 새로 생성 | - | 계획된 운영 작업 (업그레이드, 스케일링) |
| `flink cancel` | - | 마지막 것 보존 | 긴급 중지, 불필요한 Job 정리 |

> 💡 **Job이 중지된 동안에도 Generator는 계속 이벤트를 생산 중입니다.**
> 이벤트는 Kafka에 계속 쌓이고 있습니다. Flink가 복원되면 밀린 이벤트를 모두 처리합니다.

In [ ]:

# === Savepoint 생성 + Job 중지 흐름 ===

# 1. 현재 상태
#
#   Generator ──→ Kafka ──→ [Flink Job: RUNNING] ──→ PostgreSQL
#                (쌓임)     (처리 중)                (결과 저장)

# 2. Savepoint + Stop
#
#   Generator ──→ Kafka ──→ [Flink Job: STOPPED] ──→ PostgreSQL
#                (계속 쌓임) (💾 Savepoint 저장)      (더 이상 저장 안 됨)

# 3. 복원 후
#
#   Generator ──→ Kafka ──→ [Flink Job: RUNNING] ──→ PostgreSQL
#                (밀린 것    (Savepoint에서 복원)     (밀린 것까지
#                  모두 읽음)                          모두 저장)

### 3.5 Savepoint에서 복원

저장해둔 Savepoint 경로를 사용하여 Job을 재시작합니다.

```bash
# Savepoint에서 Job 재시작
docker compose exec jobmanager flink run \
  -s /tmp/flink-savepoints/savepoint-abc123-1234567890 \
  -d \
  -py /opt/flink/src/flink_job.py
```

| 옵션 | 설명 |
|------|------|
| `-s <path>` | Savepoint 경로 (여기서부터 복원) |
| `-d` | Detached 모드 (백그라운드 실행) |
| `-py <file>` | 실행할 Python 파일 |

> 📌 `-s` 옵션이 핵심입니다. 이 옵션 없이 실행하면 **처음부터 새로 시작**합니다.

### 3.6 결과 검증: 데이터 유실 없이 이어서 처리

```bash
# 복원 후 1분 대기, 그리고 결과 확인
docker compose exec postgres psql -U postgres -d streamdb \
  -c "SELECT COUNT(*) AS row_count, SUM(total_amount) AS grand_total FROM flink_results;"
```

**확인 포인트:**
- `row_count`: 이전보다 증가 → Job 중지 동안 Kafka에 쌓인 이벤트가 처리됨
- `grand_total`: 이전 값 + 새로운 이벤트 금액 → **데이터 유실 없음!**

> 💡 **핵심**: Savepoint 복원 시 Flink는 **Kafka offset도 함께 복원**합니다.
> 즉, Savepoint 시점 이후의 모든 이벤트를 Kafka에서 다시 읽어서 처리합니다.
> 이것이 **Exactly-Once 보장의 핵심**입니다.

In [ ]:

# === Savepoint 실습 전체 명령어 요약 ===

# 1. Job ID 확인
docker compose exec jobmanager flink list

# 2. 현재 결과 확인 (복원 전 스냅샷)
docker compose exec postgres psql -U postgres -d streamdb \
  -c "SELECT COUNT(*), SUM(total_amount) FROM flink_results;"

# 3. Savepoint 생성 + Job 중지
docker compose exec jobmanager flink stop \
  --savepointPath /tmp/flink-savepoints <JOB_ID>

# 4. (잠시 대기 — Kafka에 이벤트 쌓이는 중)

# 5. Savepoint에서 복원
docker compose exec jobmanager flink run \
  -s /tmp/flink-savepoints/savepoint-<...> \
  -d \
  -py /opt/flink/src/flink_job.py

# 6. 결과 확인 (밀린 이벤트까지 처리 완료)
docker compose exec postgres psql -U postgres -d streamdb \
  -c "SELECT COUNT(*), SUM(total_amount) FROM flink_results;"

---
## 4. 스케일링 실습

### 4.1 Parallelism 개념

**Parallelism** = 동시에 몇 개의 Task가 데이터를 **병렬로 처리**하는지

| Parallelism | 의미 |
|-------------|------|
| 1 | 단일 Task가 모든 데이터를 순차 처리 |
| 2 | 2개 Task가 데이터를 나눠서 병렬 처리 |
| N | N개 Task가 병렬 처리 (TaskManager의 slot 수 이하) |

In [ ]:

# === Parallelism 시각화 ===

# 📌 Parallelism = 1 (현재 기본값):
#
#   Kafka ──→ [Task 1: Source + Window + Sink] ──→ PostgreSQL
#            (모든 user 처리)
#
#   → 단일 Task가 모든 user_id의 이벤트를 처리
#   → 간단하지만, 트래픽 증가 시 병목

# 📌 Parallelism = 2:
#
#   Kafka ──┬→ [Task 1: U1, U3, U5 처리] ──┬→ PostgreSQL
#           │                                │
#           └→ [Task 2: U2, U4 처리]    ────┘
#
#   → user_id의 해시값에 따라 자동 분배 (Key Group)
#   → 같은 user_id는 항상 같은 Task가 처리 (상태 일관성)
#   → 처리량 약 2배

# 📌 Parallelism = 4:
#
#   Kafka ──┬→ [Task 1: U1, U5 처리]  ──┬→ PostgreSQL
#           ├→ [Task 2: U2 처리]      ──┤
#           ├→ [Task 3: U3 처리]      ──┤
#           └→ [Task 4: U4 처리]      ──┘
#
#   → TaskManager 2개 x 2 slots = 최대 4 병렬
#   → 처리량 약 4배

### 4.2 현재 stream-lab 설정 확인

`compose.yml`에서 TaskManager의 slot 수를 확인합니다:

```yaml
# compose.yml
taskmanager:
  environment:
    FLINK_PROPERTIES: |
      jobmanager.rpc.address: jobmanager
      taskmanager.numberOfTaskSlots: 2
```

| 구성 | 값 |
|------|-----|
| TaskManager 수 | 1개 |
| Task Slot 수 (per TM) | 2개 |
| **최대 Parallelism** | **1 x 2 = 2** |

> 💡 **Task Slot**은 TaskManager가 동시에 실행할 수 있는 Task의 수입니다.
> CPU 코어와 비슷한 개념으로 이해하면 됩니다.

### 4.3 Savepoint → Parallelism 변경 → 재시작

Parallelism을 변경하려면 반드시 **Savepoint를 생성한 후** Job을 재시작해야 합니다.
(실행 중인 Job의 Parallelism은 변경할 수 없습니다.)

**Step 1: Savepoint 생성 + Job 중지**

```bash
# 현재 Job ID 확인
docker compose exec jobmanager flink list

# Savepoint 생성 + Job 중지
docker compose exec jobmanager flink stop --savepointPath /tmp/flink-savepoints <JOB_ID>
```

**Step 2: Parallelism 변경하여 재시작**

```bash
# Parallelism 2로 변경하여 재시작
docker compose exec jobmanager flink run \
  -s <SAVEPOINT_PATH> \
  -p 2 \
  -d \
  -py /opt/flink/src/flink_job.py
```

| 옵션 | 설명 |
|------|------|
| `-s <path>` | Savepoint 경로 |
| `-p 2` | **Parallelism을 2로 설정** |
| `-d` | 백그라운드 실행 |
| `-py <file>` | 실행할 Python 파일 |

**Step 3: Web UI에서 확인**

- http://localhost:8081 → Running Jobs → Job 선택 → **Job Graph** 확인
- 각 연산자(Source, Window, Sink)의 Parallelism이 **2**로 표시
- **Subtasks** 탭에서 2개의 subtask가 병렬로 처리 중인 것을 확인

In [ ]:

# === 스케일링 절차 요약 ===

# 현재: Parallelism = 1
#
#   [Flink Job: p=1] ──→ 처리량 X
#
# 스케일 업: Parallelism = 2
#
#   Step 1. Savepoint 생성 + Job 중지
#          flink stop --savepointPath /tmp/flink-savepoints <JOB_ID>
#
#   Step 2. -p 2 옵션으로 재시작
#          flink run -s <SP> -p 2 -d -py flink_job.py
#
#   [Flink Job: p=2] ──→ 처리량 2X
#
# 💡 핵심: Savepoint가 있어야 상태를 유지하면서 스케일링 가능!
#    Savepoint 없이 -p 2로 새로 시작하면? → 이전 상태 유실!

### 4.4 TaskManager 추가 (스케일 아웃)

현재 TaskManager 1개 x 2 slots = 최대 Parallelism 2입니다.
더 높은 Parallelism이 필요하면 **TaskManager를 추가**합니다.

```bash
# TaskManager 인스턴스를 2개로 스케일
docker compose up -d --scale taskmanager=2
```

| 구성 | 변경 전 | 변경 후 |
|------|---------|---------|
| TaskManager 수 | 1개 | **2개** |
| Task Slot 수 (per TM) | 2개 | 2개 |
| **최대 Parallelism** | **2** | **4** |

**Web UI에서 확인:**
- http://localhost:8081 → **Task Managers** 탭
- TaskManager가 2개로 표시
- Free Slots 확인


![](https://cdn.discordapp.com/attachments/1457516082071081045/1470661853649834075/image.png?ex=698c1c15&is=698aca95&hm=9fa581ca4a7f4765d709e91387060e81155beb752013b128ed9d4399e4639874&)

---
## 5. 장애 복구 시나리오

Flink의 장애 복구 능력을 실제로 확인해보겠습니다.

### 5.1 TaskManager 장애 시뮬레이션

TaskManager를 강제로 재시작하여 장애 상황을 만듭니다.

```bash
# TaskManager 강제 재시작 (장애 시뮬레이션)
docker compose restart taskmanager
```

> 📌 `restart`는 컨테이너를 중지했다가 다시 시작합니다.
> 실제 운영에서 서버 크래시, OOM Kill 등과 유사한 상황입니다.

### 5.2 자동 복구 과정 관찰

**Web UI에서 관찰:**
1. http://localhost:8081 → Running Jobs
2. Job 상태: `RUNNING` → `RESTARTING` → `RUNNING`
3. Checkpoints 탭: **Restored** 체크포인트 확인

**로그에서 확인:**

```bash
# JobManager 로그에서 복구 과정 확인
docker compose logs jobmanager 2>&1 | grep -iE "restore|recover|checkpoint" | tail -10
```

예상 로그:
```
Restoring job abc123... from latest valid checkpoint
Successfully restored from checkpoint ID 42
Restarting all tasks from checkpoint
```

```bash
# TaskManager 로그에서 상태 복원 확인
docker compose logs taskmanager 2>&1 | grep -iE "restore|recover|state" | tail -10
```


# ![](https://cdn.discordapp.com/attachments/1457516082071081045/1470662229908127866/AOI_d_-1BMo58I454G2sLcXiUZ7DbJ8g0j_n4QXAA86JFPMq-AOq8ok13els1hWeYKliUni2w7HJBKkBgMTXQ8fFOErU1jNMQaLW2OsihlNzqQM7xUzgkv5-KLNcCXs_DQBslzV8oXFt2IsjJ5RrIokvbOWNds9TtQ78tCmv-cfUPu8b9imvMgs1024-rj.png?ex=698f682f&is=698e16af&hm=97f853aadd73d422f105f4a4c505976372278e6f11bc457fc67893bf9f0d2fc8&)

### 5.3 복구 후 결과 확인

```bash
# 장애 복구 후 결과 확인 (1분 대기 후)
docker compose exec postgres psql -U postgres -d streamdb \
  -c "SELECT COUNT(*) AS row_count, SUM(total_amount) AS grand_total FROM flink_results;"
```

**확인 포인트:**
- 장애 전후로 `grand_total` 정합성 유지
- 중복 데이터 없음 (Exactly-Once)
- 유실 데이터 없음 (Checkpoint에서 복원 + Kafka replay)

### 5.4 JobManager 장애 시나리오 (심화)

TaskManager와 달리 **JobManager가 죽으면** 어떻게 될까요?

```bash
# JobManager 재시작
docker compose restart jobmanager
```

| 구분 | TaskManager 장애 | JobManager 장애 |
|------|-----------------|----------------|
| **자동 복구** | ✅ 자동 (Checkpoint에서) | ❌ 수동 필요 |
| **이유** | JobManager가 살아있어서 복구 조율 | 조율자 자체가 죽음 |
| **복구 방법** | 아무것도 안 해도 됨 | Checkpoint/Savepoint에서 수동 재시작 |
| **실무 해결** | - | HA 모드 (ZooKeeper / Kubernetes) |

```bash
# JobManager 재시작 후 Job 수동 복원
# 1. 마지막 Checkpoint 경로 확인 (컨테이너 내부)
docker compose exec jobmanager ls /tmp/flink-checkpoints/

# 2. 가장 최근 Checkpoint에서 복원
docker compose exec jobmanager flink run \
  -s /tmp/flink-checkpoints/<checkpoint-dir> \
  -d \
  -py /opt/flink/src/flink_job.py
```

> 💡 **실무에서는**: Kubernetes + Flink Operator를 사용하면 JobManager 장애도 자동 복구됩니다.
> JobManager를 StatefulSet으로 배포하고, HA(High Availability) 모드를 활성화하면
> Standby JobManager가 자동으로 인수인계합니다.

In [ ]:

# === JobManager vs TaskManager 장애 비교 ===

# 📌 TaskManager 장애 → 자동 복구
#
#   JobManager(살아있음): "TaskManager가 죽었네. Checkpoint에서 복구하자."
#                          ↓
#                  새 TaskManager에 Task 재배치
#                          ↓
#                  마지막 Checkpoint에서 상태 복원
#                          ↓
#                  RUNNING 상태로 복귀 ✅

# 📌 JobManager 장애 → 수동 복구 (HA 미설정 시)
#
#   JobManager(죽음): "..."
#   TaskManager: "JobManager 연결이 끊겼어... 나도 멈출게."
#                          ↓
#                  모든 Task 중지
#                          ↓
#                  관리자가 수동으로 Job 재제출 필요
#                          ↓
#                 flink run -s <checkpoint> -d -py flink_job.py

# 📌 JobManager HA 모드 (Kubernetes 환경)
#
#   Primary JobManager(죽음) → Standby JobManager 자동 승격 → Job 자동 복구 ✅

---
## 6. 운영 체크리스트 정리

실무에서 Flink Job을 운영할 때 자주 수행하는 작업들을 표로 정리합니다.

| 운영 작업 | 방법 | 명령어 |
|-----------|------|--------|
| **장애 복구** | Checkpoint (자동) | `env.enable_checkpointing(10000)` |
| **코드 업데이트** | Savepoint → 새 코드로 재시작 | `flink stop --savepointPath ... <JOB_ID>` |
| **스케일 업** | Savepoint → -p N으로 재시작 | `flink run -s <SP> -p 2 -d -py ...` |
| **스케일 아웃** | TaskManager 추가 | `docker compose up -d --scale taskmanager=2` |
| **상태 확인** | Web UI | http://localhost:8081 |
| **Job 목록** | CLI | `flink list` |
| **로그 확인** | Docker 로그 | `docker compose logs jobmanager` |
| **Job 취소** | CLI | `flink cancel <JOB_ID>` |

In [ ]:

# === 운영 시나리오별 절차 ===

# 📌 시나리오 1: 코드 업데이트 (flink_job.py 수정)
#
#   1. flink stop --savepointPath /tmp/flink-savepoints <JOB_ID>
#   2. flink_job.py 수정 (코드 변경)
#   3. flink run -s <SAVEPOINT_PATH> -d -py /opt/flink/src/flink_job.py
#   → 상태 유지 + 새 코드 적용!

# 📌 시나리오 2: 트래픽 증가 대응
#
#   1. docker compose up -d --scale taskmanager=2   (TM 추가)
#   2. flink stop --savepointPath /tmp/flink-savepoints <JOB_ID>
#   3. flink run -s <SAVEPOINT_PATH> -p 4 -d -py /opt/flink/src/flink_job.py
#   → 4배 처리량으로 스케일 업!

# 📌 시나리오 3: 장애 발생 (TaskManager 크래시)
#
#   → 아무것도 안 해도 됨!
#   → Flink가 Checkpoint에서 자동 복구

---
## 7. Checkpoint 고급 설정 (참고)

`flink_job.py`에서 이미 설정된 내용과 추가로 고려할 수 있는 옵션들입니다.

### 7.1 현재 설정 (flink_job.py)

```python
# 기본 설정 — 이미 적용됨
env.enable_checkpointing(10_000)                                      # 10초마다 자동 저장
conf.set_string("state.backend.type", "rocksdb")                      # RocksDB 상태 백엔드
conf.set_string("state.checkpoints.dir", "file:///tmp/flink-checkpoints")
conf.set_string("execution.checkpointing.mode", "EXACTLY_ONCE")       # 정확히 한 번 보장
conf.set_string(
    "execution.checkpointing.externalized-checkpoint-retention",
    "RETAIN_ON_CANCELLATION",                                          # 취소 시 보존
)
```

### 7.2 추가 고려 옵션

```python
# Checkpoint 최소 간격: 연속 Checkpoint 사이의 최소 시간
conf.set_string("execution.checkpointing.min-pause", "5000")      # 최소 5초 간격

# Checkpoint 타임아웃: 이 시간 내에 완료되지 않으면 실패 처리
conf.set_string("execution.checkpointing.timeout", "60000")       # 60초 타임아웃

# 보관할 Checkpoint 수: 최근 N개만 유지, 나머지 삭제
conf.set_string("state.checkpoints.num-retained", "3")            # 최근 3개 보관
```

### 7.3 설정 가이드

| 설정 | 설명 | 권장값 | 주의사항 |
|------|------|--------|---------|
| `interval` | Checkpoint 주기 | 1~5분 (상태 크기에 따라) | 너무 짧으면 오버헤드 ↑ |
| `mode` | EXACTLY_ONCE / AT_LEAST_ONCE | EXACTLY_ONCE (기본) | AT_LEAST_ONCE는 더 빠르지만 중복 가능 |
| `min-pause` | 연속 Checkpoint 최소 간격 | interval의 50% | Checkpoint 생성이 겹치는 것을 방지 |
| `timeout` | Checkpoint 타임아웃 | 10분 | 상태 크기가 큰 경우 충분히 설정 |
| `num-retained` | 보관할 Checkpoint 수 | 2~3 | 디스크 공간 절약 vs 복구 유연성 |

> 💡 **실무 팁**: 개발 환경에서는 10초 간격으로 빠르게 테스트하지만,
> 프로덕션에서는 **상태 크기와 네트워크 대역폭**을 고려하여 1~5분 간격이 일반적입니다.
> 상태 크기가 GB 단위라면 Checkpoint 생성에 수십 초가 걸릴 수 있습니다.

In [ ]:

# === State Backend 비교 ===

# 📌 HashMap (기본)
#
#   ┌────────────────────────────┐
#   │        JVM Heap Memory      │
#   │   ┌───────────────────┐    │
#   │   │  State (HashMap)  │    │
#   │   │  Key → Value      │    │
#   │   └───────────────────┘    │
#   └────────────────────────────┘
#
#   ✅ 빠름 (메모리 직접 접근)
#   ❌ 메모리 제한 (JVM Heap 크기)
#   ❌ 대용량 상태 불가
#   → 용도: 상태가 작은 경우 (수 MB ~ 수 GB)

# 📌 RocksDB (stream-lab에서 사용 중)
#
#   ┌────────────────────────────┐
#   │        JVM Heap Memory      │
#   │   ┌─────────────────┐      │
#   │   │  Cache (일부)    │      │
#   │   └────────┬────────┘      │
#   │            │ 읽기/쓰기      │
#   │   ┌────────▼────────┐      │
#   │   │   RocksDB (Disk) │      │
#   │   │   Key → Value    │      │
#   │   └─────────────────┘      │
#   └────────────────────────────┘
#
#   ✅ 대용량 상태 가능 (디스크 기반)
#   ✅ 증분 Checkpoint 지원
#   ❌ HashMap보다 느림 (직렬화/역직렬화)
#   → 용도: 상태가 큰 경우 (수 GB ~ 수 TB)

---
## 📝 FAQ

**Q1. Checkpoint 주기를 너무 짧게 설정하면?**

→ **오버헤드 증가**, 처리 성능 저하. Checkpoint 생성에는 상태를 직렬화하고 디스크에 기록하는 비용이 발생합니다. 상태 크기가 크면 Checkpoint 간격보다 Checkpoint 생성 시간이 더 길어져서, Checkpoint가 연속적으로 겹치는 문제가 발생할 수 있습니다. `min-pause` 설정으로 이를 방지할 수 있습니다.

**Q2. Savepoint 없이 코드를 업데이트하면?**

→ **이전 상태 유실**. Job을 cancel하고 새로 시작하면, Flink는 처음부터 다시 시작합니다. `scan.startup.mode = latest-offset`이므로 이전 이벤트는 다시 읽을 수 없고, 윈도우 집계 상태도 사라집니다. 반드시 Savepoint를 생성한 후 업데이트하세요.

**Q3. Parallelism을 변경하면 상태가 어떻게 분배되나요?**

→ **Key Group 기반으로 자동 재분배**됩니다. Flink는 내부적으로 상태를 Key Group이라는 단위로 관리합니다. Parallelism이 변경되면 Key Group이 새로운 Task에 자동으로 재배치됩니다. 개발자가 직접 분배 로직을 작성할 필요가 없습니다.

**Q4. RocksDB vs HashMap 상태 백엔드, 어떤 것을 선택해야 하나요?**

| 기준 | HashMap | RocksDB |
|------|---------|---------|
| 상태 크기 | 수 MB ~ 수 GB | 수 GB ~ 수 TB |
| 접근 속도 | 빠름 (메모리) | 느림 (디스크) |
| 증분 Checkpoint | ❌ | ✅ |
| 메모리 부담 | 높음 (전체 상태가 Heap) | 낮음 (캐시만 Heap) |
| 권장 상황 | 소규모, 낮은 지연 | 대규모, 안정적 운영 |

→ **실무에서는 대부분 RocksDB를 사용**합니다. 상태 크기가 예측하기 어렵기 때문입니다.

**Q5. 실무에서 Checkpoint 저장소로 뭘 쓰나요?**

→ **HDFS, S3, GCS 등 분산 스토리지**를 사용합니다. 로컬 파일시스템(`file:///`)은 개발 환경에서만 사용합니다. 분산 스토리지를 사용하면 JobManager나 TaskManager가 어느 노드에서 재시작되어도 Checkpoint에 접근할 수 있습니다.

```python
# 실무 설정 예시
conf.set_string("state.checkpoints.dir", "s3://my-bucket/flink-checkpoints")
conf.set_string("state.savepoints.dir", "s3://my-bucket/flink-savepoints")
```

---
## ✅ 퀴즈

**Q1.** Checkpoint와 Savepoint의 가장 큰 차이는 무엇인가요?

- A) Checkpoint는 디스크에, Savepoint는 메모리에 저장한다
- B) Checkpoint는 자동 생성, Savepoint는 수동 생성이다
- C) Checkpoint는 상태를, Savepoint는 코드를 저장한다
- D) Checkpoint는 빠르고, Savepoint는 느리다

<details>
<summary>정답 확인</summary>

**정답: B)**

Checkpoint는 Flink가 **자동으로 주기적으로** 생성하며, 주로 **장애 복구** 용도입니다.
Savepoint는 사용자가 **수동으로 원하는 시점에** 생성하며, **운영 작업**(코드 업데이트, 스케일 변경 등) 용도입니다.
둘 다 상태를 디스크에 저장하며, 형식도 유사합니다.

</details>

---

**Q2.** TaskManager가 크래시된 후 자동으로 복구되는 이유는 무엇인가요?

- A) Kafka가 이벤트를 다시 보내주기 때문
- B) PostgreSQL에 결과가 이미 저장되어 있기 때문
- C) JobManager가 마지막 Checkpoint에서 상태를 복원하기 때문
- D) Docker가 컨테이너를 자동으로 재시작하기 때문

<details>
<summary>정답 확인</summary>

**정답: C)**

**JobManager**가 살아있으므로, TaskManager 장애를 감지하고 **마지막 성공 Checkpoint에서 상태를 복원**합니다.
복원 후 Kafka에서 Checkpoint 시점 이후의 이벤트를 다시 읽어 처리합니다.
A)의 Kafka replay도 복구 과정의 일부이지만, 핵심은 Checkpoint 기반의 상태 복원입니다.

</details>

---

**Q3.** 실행 중인 Flink Job의 Parallelism을 2에서 4로 변경하려면 어떤 절차가 필요한가요?

- A) Web UI에서 Parallelism 슬라이더를 조절한다
- B) `flink modify <JOB_ID> -p 4` 명령어를 실행한다
- C) Savepoint 생성 → Job 중지 → `-p 4` 옵션으로 재시작한다
- D) `compose.yml`에서 `numberOfTaskSlots`를 변경하고 컨테이너를 재시작한다

<details>
<summary>정답 확인</summary>

**정답: C)**

실행 중인 Job의 Parallelism은 **동적으로 변경할 수 없습니다.**
반드시 **Savepoint를 생성**하여 현재 상태를 보존한 후, Job을 **중지**하고,
`-p 4` 옵션을 추가하여 **새로 시작**해야 합니다.
또한 최대 Parallelism 4를 지원하려면 TaskManager의 총 slot 수도 4 이상이어야 합니다.

</details>

---
## 📌 핵심 요약

| 개념 | 핵심 내용 |
|------|----------|
| **Checkpoint** | 자동 저장, 장애 복구용, Flink 내부 관리, 주기적 생성 |
| **Savepoint** | 수동 저장, 운영 작업용, 사용자 관리, 영구 보존 |
| **Exactly-Once** | Checkpoint + Kafka offset 복원으로 **정확히 한 번** 처리 보장 |
| **Parallelism** | Task 병렬도, Savepoint 생성 후 `-p N` 옵션으로 변경 가능 |
| **State Backend** | RocksDB (디스크, 대용량) vs HashMap (메모리, 소규모) |
| **TaskManager 장애** | 마지막 Checkpoint에서 **자동 복원**, 최대 interval만큼 재처리 |
| **JobManager 장애** | 수동 복원 필요, 실무에서는 HA 모드로 자동화 |
| **스케일링** | Savepoint → 중지 → Parallelism/TM 변경 → 재시작 |

---
## 🏁 Day28 마무리

### 오늘 배운 것

| 교시 | 주제 | 핵심 |
|------|------|------|
| 1~2교시 | stream-lab Phase 1~4 | Processing Time의 한계 → Flink(Event Time)로 해결 |
| 3교시 | Fraud Detection | DataStream API로 복잡한 이벤트 패턴 탐지 |
| **4교시** | **Flink 운영** | **Checkpoint, Savepoint, 스케일링, 장애 복구** |

### 데이터 엔지니어링 파이프라인 전체 그림

In [ ]:

# === Day27-28 전체 여정 ===

# Day27:
#   "왜 Flink인가?" → Spark(Micro-batch) vs Flink(Native Streaming)
#   "어떻게 쓰는가?" → PyFlink Table API + DataStream API
#
# Day28:
#   "실제로 해보자" → stream-lab Phase 1~4
#                     Python(Processing Time)의 한계 → Flink(Event Time)로 해결
#
#   "복잡한 처리" → Fraud Detection (DataStream API)
#                   상태 관리 + 패턴 매칭 + Side Output
#
#   "운영해보자" → Checkpoint, Savepoint, 스케일링, 장애 복구
#                  코드를 배포하는 것과 운영하는 것의 차이

# === 데이터 엔지니어링 파이프라인에서의 위치 ===

#   [수집]          [전달]         [처리]              [저장]         [활용]
#   Producer ──→ Kafka Broker ──→ Flink Job     ──→ PostgreSQL ──→ Dashboard
#                                  │                                   API
#                                  │ Checkpoint                        ML Model
#                                  │ Savepoint
#                                  │ Scaling
#                                  └→ 이 모든 것이 "운영"

# 💡 결론:
#    데이터 엔지니어링은 구현 이후의 운영 단계까지 포함합니다.
#    Flink의 Checkpoint, Savepoint, 스케일링은 상태 일관성과 서비스 연속성을 보장하는 핵심 도구입니다.